# Quick Reference Setup

The table below summarises the main design choices used in this submission. These settings are fixed before the official out-of-sample backtest is evaluated.

| Item | This notebook |
|---|---|
| Frequency | Weekly Friday close (`W-FRI`), rebalance weekly |
| Universe | 45 liquid U.S. large-cap stocks across 9 GICS-aligned sectors |
| Sample period | Raw data from 2016-01-01 to 2026-04-25; modelling sample from 2017-01-06 to 2026-04-17 |
| Official OOS holdout | 104 weekly decision dates, from 2024-04-26 to 2026-04-17 |
| Validation block | 26-week pre-holdout validation block, from 2023-10-27 to 2024-04-19 |
| Model refit | Monthly expanding-window refit during the official holdout |
| Models | Logistic Regression and Random Forest |
| Portfolio | Long top 20%, short bottom 20%, equal-weight within each leg |
| Transaction cost | 5 bps one-way base case; 1 bps and 10 bps sensitivity checks |
| Benchmark | SPY buy-and-hold and equal-weight universe |
| Turnover | Drifted one-way turnover over the full 45-stock vector |
| Risk control | 10% annualised volatility target with scaler clipped to `[0.5, 1.5]` |
| Data source | `yfinance` only; no FRED macro series downloaded |
| Random seed | `random_state = 42` where applicable |

The full code, tables, figures, and checks are generated in the sections below.

# Machine-Learning Equity Selection and Backtesting

---

## 0a) Installation

This analysis uses **yfinance** as the sole market-data source. Weekly SPY returns serve as the market-state proxy. No FRED series are downloaded.


In [ ]:
!pip install -q yfinance --upgrade
print("yfinance installation complete.")

## 0b) Imports and Random Seeds

This section defines the computational environment, random seed, output paths, and shared utility functions.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os, math, time, random, warnings, datetime
from pathlib import Path
from itertools import product

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import yfinance as yf

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.exceptions       import ConvergenceWarning

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot  as plt
import matplotlib.dates   as mdates

# ── Notebook display ─────────────────────────────────────────────────────────
from IPython.display import Image as _IPImage, display as _ipy_display, HTML as _HTML

# ── Suppress non-critical warnings ───────────────────────────────────────────
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ── Version log ───────────────────────────────────────────────────────────────
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"yfinance   : {yf.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"matplotlib : {matplotlib.__version__}")

# All stochastic objects receive random_state=RANDOM_SEED explicitly.
# No model uses an implicit random state.
RANDOM_SEED: int = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Random seed set: {RANDOM_SEED}")

OUTPUT_DIR: Path = Path("./outputs")
TABLE_DIR:  Path = OUTPUT_DIR / "tables"
FIG_DIR:    Path = OUTPUT_DIR / "figures"

for _d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Output directories ready:")
print(f"  Tables  : {TABLE_DIR.resolve()}")
print(f"  Figures : {FIG_DIR.resolve()}")

This section defines the computational environment, random seed, output paths, and shared helper functions.

In [ ]:
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    """Save DataFrame to TABLE_DIR/<name>.csv and print confirmation.
    Pass index=True only when the index carries meaningful information.
    """
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"  [saved] {path.name}  ({len(df)} rows × {len(df.columns)} cols)")
    return path


def save_figure(fig: plt.Figure, name: str, dpi: int = 150) -> Path:
    """Save Figure to FIG_DIR/<name>.png, close it, and print confirmation."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [saved] {path.name}")
    return path


print("Utility functions defined: save_table() (index=False default), save_figure()")

plt.rcParams.update({
    "figure.figsize"  : (11, 5),
    "figure.dpi"      : 100,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "axes.grid"       : True,
    "grid.alpha"      : 0.35,
    "grid.linestyle"  : "--",
    "font.size"       : 11,
    "axes.titlesize"  : 12,
    "axes.labelsize"  : 11,
    "legend.fontsize" : 10,
    "legend.framealpha": 0.8,
    "lines.linewidth" : 1.6,
})

# Colour palette
PALETTE = {
    "lr"      : "#2166ac",   # blue   — Logistic Regression net
    "rf"      : "#d6604d",   # red    — Random Forest net
    "spy"     : "#4dac26",   # green  — SPY buy-and-hold
    "ew"      : "#756bb1",   # purple — Equal-weight universe
    "lr_gross": "#92c5de",   # light blue — LR gross (diagnostic only)
    "rf_gross": "#f4a582",   # light red  — RF gross (diagnostic only)
}

print("Matplotlib style configured.")

def show_table(df: pd.DataFrame, max_rows: int = None, index: bool = False) -> None:
    table_df = df.copy()
    if max_rows is not None:
        table_df = table_df.head(max_rows)
    _ipy_display(_HTML(table_df.to_html(index=index, escape=False)))


## 1) User Settings

### Investment Problem

This notebook tests whether weekly machine-learning scores can be turned into a tradable cross-sectional equity strategy. The universe contains 45 large-cap U.S. stocks across 9 sectors. Each week, Logistic Regression and Random Forest models produce predicted probabilities for next-week positive returns. These scores are converted into a long-short portfolio by buying the top-ranked stocks and shorting the bottom-ranked stocks.

The evaluation focuses on whether the strategy still looks useful after realistic frictions: walk-forward refitting, transaction costs, turnover, benchmark comparison, volatility targeting, and robustness checks. The main goal is not just prediction accuracy, but whether the signal survives as a portfolio strategy.

**Timing convention:** all features use information available by Friday close at date t. The target is the return over week t+1. Portfolio weights are formed at date t and evaluated using the next-week return. This convention is used throughout the notebook to avoid look-ahead bias.

All strategy parameters are defined here. These values are fixed before any backtest is run and are not changed after examining holdout results.

In [ ]:
# ── Data parameters ───────────────────────────────────────────────────────────
DATA_START: str       = "2016-01-01"   # raw download start
DATA_END:   str       = "2026-04-25"   # fixed endpoint (Friday) for reproducibility
RESAMPLE_FREQ: str    = "W-FRI"        # weekly Friday-close
PERIODS_PER_YEAR: int = 52

# ── Holdout / validation ──────────────────────────────────────────────────────
N_HOLDOUT_WEEKS:    int = 104   # official out-of-sample holdout
N_VALIDATION_WEEKS: int = 26    # one-time pre-holdout validation block

# ── Portfolio construction ─────────────────────────────────────────────────────
# Official A2 brief recommends long top quintile / short bottom quintile.
# Robustness checks will test q = 0.10 and q = 0.30.
BASE_QUANTILE: float = 0.20

# ── Volatility targeting ──────────────────────────────────────────────────────
# Conservative settings aligned with course Week 9/10 and teacher guidance.
# Set before any holdout results were examined.
VOL_TARGET_ANN: float = 0.10         # 10% annualised volatility target
VOL_WINDOW:     int   = 26           # trailing window for realised vol
SCALER_CLIP:    tuple = (0.5, 1.5)   # prevent extreme leverage changes

# ── Transaction costs ─────────────────────────────────────────────────────────
BASE_COST_BPS:         float = 5.0
SENSITIVITY_COSTS_BPS: list  = [1.0, 5.0, 10.0]

# ── Feature windows ───────────────────────────────────────────────────────────
MOM_SHORT_WINDOW:  int = 4
MOM_LONG_WINDOW:   int = 12
VOL_SHORT_WINDOW:  int = 8
VOL_LONG_WINDOW:   int = 26
MA_SHORT_WINDOW:   int = 8
MA_LONG_WINDOW:    int = 26
DRAWDOWN_WINDOW:   int = 52
VOL_CHG_WINDOW:    int = 4
ADV_WINDOW:        int = 12

# ── Model tuning grids ────────────────────────────────────────────────────────
LR_C_GRID:          list = [0.3, 1.0, 3.0]
RF_MAX_DEPTH_GRID:  list = [4, 6, 8]
RF_MIN_LEAF_GRID:   list = [5, 10, 20]
RF_N_ESTIMATORS:    int  = 300      # fixed; not tuned
RF_MAX_FEATURES:    str  = "sqrt"

# ── Missingness tracking (populated across Sections 4–7) ───────────────────────
_n_tickers_attempted: int   = 45    # updated from len(TICKERS) in Section 3
_n_rows_raw:          int   = 0     # set in Section 5
_n_rows_after_ticker: int   = 0     # set in Section 5
_n_rows_after_price:  int   = 0     # set in Section 5
_n_rows_after_burnin: int   = 0     # set in Sections 6/7
_n_rows_after_target: int   = 0     # set in Section 7
_n_burnin_dropped:    int   = 0     # set in Section 7
_n_terminal_dropped:  int   = 0     # set in Section 7
_failed_ticker_log:   list  = []    # (ticker, reason) tuples; set in Section 5

print("Global constants defined.")
print(f"  Data window     : {DATA_START} → {DATA_END}")
print(f"  Holdout         : {N_HOLDOUT_WEEKS} weeks")
print(f"  Validation      : {N_VALIDATION_WEEKS} weeks (one-time pre-holdout block)")
print(f"  Base quantile   : top/bottom {int(BASE_QUANTILE*100)}%")
print(f"  Vol target      : {VOL_TARGET_ANN*100:.0f}% ann., scaler clip {SCALER_CLIP}")
print(f"  Base cost       : {BASE_COST_BPS} bps one-way")
print(f"  Random seed     : {RANDOM_SEED}")

45-stock universe: 9 GICS-aligned sectors, 5 names each. Manually defined for reproducibility.

In [ ]:
# ── Ticker list (45 stocks, 9 sectors × 5 each) ──────────────────────────────
TICKERS: list = [
    # Technology (5)
    "AAPL", "MSFT", "NVDA", "AVGO", "AMD",
    # Communication Services (5) — GOOGL and META are NOT Technology
    "GOOGL", "META", "NFLX", "DIS", "CMCSA",
    # Healthcare (5)
    "JNJ", "UNH", "LLY", "ABBV", "MRK",
    # Consumer Staples (5)
    "PG", "KO", "PEP", "WMT", "COST",
    # Consumer Discretionary (5)
    "AMZN", "TSLA", "HD", "MCD", "NKE",
    # Financials (5)
    "JPM", "BAC", "GS", "MS", "BLK",
    # Industrials (5)
    "HON", "UPS", "CAT", "DE", "GE",
    # Energy (5)
    "XOM", "CVX", "COP", "EOG", "SLB",
    # Materials (5)
    "LIN", "APD", "NEM", "FCX", "NUE",
]

_n_tickers_attempted = len(TICKERS)   # update global tracker

assert len(TICKERS) == 45, "Universe must have exactly 45 tickers"
assert len(set(TICKERS)) == 45, "Duplicate tickers detected"
print(f"Initial universe: {len(TICKERS)} tickers")

# ── Sector mapping ───────────────────────────────────────────────────────────
SECTOR_MAP: dict = {
    # Technology
    "AAPL": "Technology",  "MSFT": "Technology",  "NVDA": "Technology",
    "AVGO": "Technology",  "AMD":  "Technology",
    # Communication Services
    "GOOGL": "Communication Services",  "META":  "Communication Services",
    "NFLX":  "Communication Services",  "DIS":   "Communication Services",
    "CMCSA": "Communication Services",
    # Healthcare
    "JNJ":  "Healthcare",  "UNH":  "Healthcare",  "LLY":  "Healthcare",
    "ABBV": "Healthcare",  "MRK":  "Healthcare",
    # Consumer Staples
    "PG":   "Consumer Staples",  "KO":   "Consumer Staples",
    "PEP":  "Consumer Staples",  "WMT":  "Consumer Staples",
    "COST": "Consumer Staples",
    # Consumer Discretionary
    "AMZN": "Consumer Discretionary",  "TSLA": "Consumer Discretionary",
    "HD":   "Consumer Discretionary",  "MCD":  "Consumer Discretionary",
    "NKE":  "Consumer Discretionary",
    # Financials
    "JPM": "Financials",  "BAC": "Financials",  "GS":  "Financials",
    "MS":  "Financials",  "BLK": "Financials",
    # Industrials
    "HON": "Industrials",  "UPS": "Industrials",  "CAT": "Industrials",
    "DE":  "Industrials",  "GE":  "Industrials",
    # Energy
    "XOM": "Energy",  "CVX": "Energy",  "COP": "Energy",
    "EOG": "Energy",  "SLB": "Energy",
    # Materials
    "LIN": "Materials",  "APD": "Materials",  "NEM": "Materials",
    "FCX": "Materials",  "NUE": "Materials",
}

assert set(SECTOR_MAP.keys()) == set(TICKERS), "SECTOR_MAP must cover all TICKERS"
print("SECTOR_MAP defined for all 45 tickers.")

## 2) Download Data

Daily adjusted close prices and share volume are downloaded via yfinance for all 45 universe tickers plus SPY. Data are then resampled to weekly Friday-close frequency. The download start date is set in `DATA_START` (section 1) for reproducibility.

Daily adjusted prices and share volume are downloaded via yfinance for all 45 universe tickers plus SPY. The raw download covers `DATA_START` to `DATA_END`.

In [ ]:
# ── Download all tickers + SPY in one call ───────────────────────────────────
_all_dl_tickers = TICKERS + ["SPY"]
print(f"Downloading {len(_all_dl_tickers)} tickers ({len(TICKERS)} universe + SPY)...")
print(f"Period: {DATA_START} → {DATA_END}")

_t0 = time.time()
raw = yf.download(
    tickers   = _all_dl_tickers,
    start     = DATA_START,
    end       = DATA_END,
    auto_adjust = True,       # Close = dividend/split-adjusted close
    progress    = True,
    threads     = True,
)
_elapsed = time.time() - _t0
print(f"\nDownload complete in {_elapsed:.1f}s.  Raw shape: {raw.shape}")

# ── Extract Close (adjusted) and Volume ──────────────────────────────────────
# yfinance returns MultiIndex columns: (field, ticker).
# auto_adjust=True means 'Close' is already the split-and-dividend-adjusted price.

if isinstance(raw.columns, pd.MultiIndex):
    close_daily  = raw["Close"].copy()
    volume_daily = raw["Volume"].copy()
else:
    # Defensive fallback (single-ticker edge case)
    close_daily  = raw[["Close"]].copy()
    volume_daily = raw[["Volume"]].copy()

# Confirm expected tickers present
_missing_from_download = [t for t in _all_dl_tickers if t not in close_daily.columns]
if _missing_from_download:
    print(f"WARNING: these tickers not found in download: {_missing_from_download}")
else:
    print(f"All {len(_all_dl_tickers)} tickers present in download.")



Daily dollar volume is computed as close × volume. All series are resampled to weekly Friday-close frequency. SPY is separated as the market benchmark series.

In [ ]:
# daily dollar volume = adjusted close × share volume
dollar_vol_daily = close_daily.multiply(volume_daily)


# ── Resample daily → weekly (Friday close) ───────────────────────────────────
# .last() picks the last trading day in each calendar week ending on Friday.
# Weeks with no trading days (holidays) will be NaN — handled in Section 5.

price_weekly       = close_daily.resample(RESAMPLE_FREQ).last()
dollar_vol_weekly  = dollar_vol_daily.resample(RESAMPLE_FREQ).sum(min_count=1)   # sum daily $ vol

# Weekly return: pct_change on Friday-close prices.
# The FIRST row per ticker will be NaN (no prior Friday close) — expected burn-in.
ret_weekly = price_weekly.pct_change()

print(f"price_weekly       : {price_weekly.shape}  "
      f"({price_weekly.index[0].date()} → {price_weekly.index[-1].date()})")
print(f"dollar_vol_weekly  : {dollar_vol_weekly.shape}")
print(f"ret_weekly         : {ret_weekly.shape}  "
      f"(first row NaN per ticker is expected)")

# ── Isolate SPY series before focusing on the 45-stock universe ──────────────
spy_price_weekly  = price_weekly["SPY"].copy()
spy_ret_weekly    = ret_weekly["SPY"].copy()

# Remove SPY from the universe matrices
price_weekly      = price_weekly.drop(columns=["SPY"])
dollar_vol_weekly = dollar_vol_weekly.drop(columns=["SPY"])
ret_weekly        = ret_weekly.drop(columns=["SPY"])

_n_weekly_dates = len(price_weekly)

print(f"SPY series    : {len(spy_ret_weekly)} weekly observations")
print(f"Universe wide : {price_weekly.shape}  (dates × tickers)")
print(f"SPY NaN count : {spy_ret_weekly.isna().sum()} "
      f"(first row NaN expected from pct_change)")

## 3) Universe Summary

The universe consists of **45 liquid U.S. large-cap equities across 9 GICS-aligned sectors**,
with exactly 5 stocks per sector.

Key design choices:
- **Sector balance** (5 per sector) avoids concentration in any single industry,
  ensuring genuine cross-sectional return variation across macro regimes.
- **Communication Services** is a distinct sector: GOOGL and META are classified here,
  not in Technology, following the 2018 GICS reclassification.
- All names are current S&P 500 large-cap constituents with histories back to at least 2016,
  providing sufficient data for the 52-week rolling feature burn-in.
- The universe was defined **before** downloading data or examining any backtest results.
- This universe differs from the teacher sample (Technology 20 + Manufacturing 20 + Financials 10)
  in both sector structure and individual stock selection.

Tickers are screened for download success and sufficient price history. Any that fail the quality gate are logged and removed before the panel is built. The liquidity summary confirms implementability; all names are S&P 500 large-caps.

The table below shows the initial sector composition of the 45-stock universe.

In [ ]:
# ── Quality screen: all 45 tickers pass ──────────────────────────────────────
# Pre-cleaning composition equals final composition because all 45 tickers
# have sufficient history (≥ 2017-01-01) and < 10% missing return rate.
# The final sector_count_table (using FINAL_TICKERS) is saved below.
print(f"Universe: {len(TICKERS)} tickers attempted.")
print("Quality checks: history start, missing rate. Results shown after screening.")

Tickers are screened against three criteria: successful download, history starting by 2017-01-01, and a missing-return rate below 10%. Any tickers that fail are removed and logged; the remaining set defines `FINAL_TICKERS`.

In [ ]:
# ── Ticker quality check ──────────────────────────────────────────────────────
MIN_HISTORY_START = pd.Timestamp("2017-01-01")
MAX_MISSING_RATE  = 0.10

_failed_ticker_log = []    # list of (ticker, reason) tuples

for ticker in TICKERS:
    # Check 1: present in download?
    if ticker not in price_weekly.columns:
        _failed_ticker_log.append((ticker, "download_failed"))
        continue

    series = price_weekly[ticker]

    # Check 2: sufficient history?
    first_valid = series.first_valid_index()
    if first_valid is None or first_valid > MIN_HISTORY_START:
        _failed_ticker_log.append(
            (ticker, f"history_starts_{first_valid.date() if first_valid else 'never'}"))
        continue

    # Check 3: missing rate?
    missing_rate = series.isna().mean()
    if missing_rate > MAX_MISSING_RATE:
        _failed_ticker_log.append(
            (ticker, f"missing_rate_{missing_rate:.1%}"))

FAILED_TICKERS = [t for t, _ in _failed_ticker_log]
FINAL_TICKERS  = [t for t in TICKERS if t not in FAILED_TICKERS]

print(f"Tickers attempted : {len(TICKERS)}")
print(f"Tickers removed   : {len(FAILED_TICKERS)}")
if FAILED_TICKERS:
    for t, reason in _failed_ticker_log:
        print(f"  REMOVED {t:8s}  reason: {reason}")
else:
    print("  (none removed — all 45 pass quality checks)")
print(f"Final universe    : {len(FINAL_TICKERS)} tickers")

# ── Restrict wide matrices to FINAL_TICKERS ──────────────────────────────────
price_weekly      = price_weekly[FINAL_TICKERS]
dollar_vol_weekly = dollar_vol_weekly[FINAL_TICKERS]
ret_weekly        = ret_weekly[FINAL_TICKERS]

# ── Track row counts for missingness audit ────────────────────────────────────
# Step 1: theoretical raw rows (all dates × all attempted tickers)
_n_rows_raw = _n_weekly_dates * _n_tickers_attempted

# Step 2: after ticker filter (all dates × final tickers)
_n_rows_after_ticker = _n_weekly_dates * len(FINAL_TICKERS)

print(f"Theoretical raw rows         (step 1): {_n_rows_raw:,}")
print(f"After ticker filter rows     (step 2): {_n_rows_after_ticker:,}")
print(f"Rows removed by ticker filter        : "
      f"{_n_rows_raw - _n_rows_after_ticker:,}")

The liquidity summary confirms that all universe names maintain adequate trading volume for implementation. The sector breakdown below reflects the final composition after quality screening.

In [ ]:
# ── Save sector_count_table using FINAL_TICKERS (official version) ───────────
sector_count_table = (
    pd.Series({t: SECTOR_MAP[t] for t in FINAL_TICKERS})
    .rename("sector")
    .reset_index()
    .rename(columns={"index": "ticker"})
    .groupby("sector")["ticker"]
    .agg(["count", lambda x: ", ".join(sorted(x))])
    .rename(columns={"count": "n_tickers", "<lambda_0>": "tickers"})
    .sort_values("n_tickers", ascending=False)
    .reset_index()
)

print(f"Final universe: {len(FINAL_TICKERS)} tickers across {len(sector_count_table)} sectors.")
save_table(sector_count_table, "sector_count_table")
show_table(sector_count_table[["sector","n_tickers"]])

## 4) Build Panel Dataset

Wide-format weekly price, volume, and return matrices are stacked into a long-format panel indexed by `(date, ticker)`. Rows with missing close, return, or dollar-volume values are dropped and counted transparently.

In [ ]:
# ── Stack wide → long ────────────────────────────────────────────────────────
# Use dropna=False to capture ALL (date, ticker) pairs including NaN rows,
# so we can count and audit missing price/volume rows explicitly.
#
# Use rename_axis(['date','ticker']) BEFORE reset_index() so that column names
# are always 'date' and 'ticker' regardless of pandas version or yfinance index
# naming (yfinance >=0.2 sets index name to 'Date'/'Datetime', not None, so
# the previous rename(columns={'level_0':...}) approach silently failed).

close_long = (price_weekly
              .stack(dropna=False)
              .rename_axis(['date', 'ticker'])
              .reset_index(name='close'))

dollar_vol_long = (dollar_vol_weekly
                   .stack(dropna=False)
                   .rename_axis(['date', 'ticker'])
                   .reset_index(name='dollar_vol'))

ret_long = (ret_weekly
            .stack(dropna=False)
            .rename_axis(['date', 'ticker'])
            .reset_index(name='ret'))

# Merge into a single panel
panel = (close_long
         .merge(dollar_vol_long, on=["date", "ticker"], how="inner")
         .merge(ret_long,        on=["date", "ticker"], how="inner"))

panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Long-format panel (before price cleaning): {panel.shape}")
print(f"  NaN close     : {panel['close'].isna().sum():,}")
print(f"  NaN dollar_vol: {panel['dollar_vol'].isna().sum():,}")
print(f"  NaN ret       : {panel['ret'].isna().sum():,}")


# ── Drop rows with missing close or ret ──────────────────────────────────────
# These arise from trading halts, delistings, or weeks with no trading days.
# The first weekly ret row per ticker is naturally NaN (pct_change burn-in);
# this is also captured here.

_before_price_clean = len(panel)
panel = panel.dropna(subset=["close", "ret", "dollar_vol"]).reset_index(drop=True)
_n_price_dropped = _before_price_clean - len(panel)
_n_rows_after_price = len(panel)

# Update global tracker
_n_rows_after_ticker = _before_price_clean   # was theoretical; update to actual
_n_rows_raw = _n_tickers_attempted * _n_weekly_dates  # keep as theoretical

print(f"Rows before price cleaning : {_before_price_clean:,}")
print(f"Rows dropped (NaN close/ret): {_n_price_dropped:,}")
print(f"Rows remaining             : {_n_rows_after_price:,}")

# ── Compute median weekly dollar volume per ticker ───────────────────────────
# We report in approximate daily terms (divide weekly sum by 5 trading days)
# to match industry convention for daily dollar volume thresholds.

liq = (panel
       .groupby("ticker")["dollar_vol"]
       .median()
       .rename("median_weekly_dollar_vol")
       .reset_index())

liq["median_daily_dollar_vol_approx"] = liq["median_weekly_dollar_vol"] / 5

liq_stats = {
    "Metric": [
        "Min median daily $ vol (approx)",
        "25th pct median daily $ vol",
        "Median median daily $ vol",
        "75th pct median daily $ vol",
        "Max median daily $ vol",
    ],
    "Value ($M)": [
        liq["median_daily_dollar_vol_approx"].min()  / 1e6,
        liq["median_daily_dollar_vol_approx"].quantile(0.25) / 1e6,
        liq["median_daily_dollar_vol_approx"].median() / 1e6,
        liq["median_daily_dollar_vol_approx"].quantile(0.75) / 1e6,
        liq["median_daily_dollar_vol_approx"].max()  / 1e6,
    ]
}

liquidity_summary_table = pd.DataFrame(liq_stats)
liquidity_summary_table["Value ($M)"] = liquidity_summary_table["Value ($M)"].round(1)
save_table(liquidity_summary_table, "liquidity_summary_table")
show_table(liquidity_summary_table)

## 5) Feature Engineering

All 11 features are computed within ticker-level groups using backward-looking rolling windows. `shift(-1)` is reserved exclusively for target construction in section 7 and does not appear in any feature calculation.

Feature categories: **Price/Momentum** (ret_t, mom_4w, mom_12w), **Risk** (vol_8w, vol_26w), **Trend** (ma_gap_8, ma_gap_26, drawdown_52w), **Volume/Liquidity** (vol_chg_4w, adv_12w_log), and **Market State** (spy_ret_t).

SPY weekly return is merged into the panel as a market-state control variable. The per-ticker feature function is then defined; all features use data dated ≤ t.

In [ ]:
# ── Merge SPY weekly return into panel ───────────────────────────────────────
# spy_ret_t is a market-level feature: same value for all tickers on date t.
# It is available by Friday close t (same observation date as all other features).
# This notebook uses yfinance only. No FRED data is downloaded.
# Market-state information is simplified using SPY weekly returns
# rather than a full point-in-time macro release database.

_spy_map = spy_ret_weekly.dropna().to_dict()
panel["spy_ret_t"] = panel["date"].map(_spy_map)

_spy_missing = panel["spy_ret_t"].isna().sum()
print(f"spy_ret_t merged.  NaN count: {_spy_missing} "
      f"(expected: ~{len(FINAL_TICKERS)} from first-week SPY NaN)")

def _compute_ticker_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all rolling features for a single ticker's time series.

    This function is applied via groupby(...).apply() to ensure each
    rolling window is computed within the ticker group, never across tickers.

    Timing: all features use data dated <= t (backward-looking only).
    shift(-1) is NOT used here; it is reserved for target construction in Section 7.

    Parameters
    ----------
    df : DataFrame for one ticker, sorted ascending by date.

    Returns
    -------
    df with feature columns added.
    """
    df = df.sort_values("date").copy()

    # ── Price / Momentum ──────────────────────────────────────────────────────
    # ret_t = current weekly return (observable at Friday close t)
    df["ret_t"]   = df["ret"]

    # mom_4w: sum of last 4 weekly returns including t
    # rolling(4).sum() at row t = ret[t-3] + ret[t-2] + ret[t-1] + ret[t]
    df["mom_4w"]  = df["ret"].rolling(MOM_SHORT_WINDOW,
                                       min_periods=MOM_SHORT_WINDOW).sum()
    df["mom_12w"] = df["ret"].rolling(MOM_LONG_WINDOW,
                                       min_periods=MOM_LONG_WINDOW).sum()

    # ── Risk ─────────────────────────────────────────────────────────────────
    # vol_8w / vol_26w: realised weekly return std over past N weeks including t
    df["vol_8w"]  = df["ret"].rolling(VOL_SHORT_WINDOW,
                                       min_periods=VOL_SHORT_WINDOW).std()
    df["vol_26w"] = df["ret"].rolling(VOL_LONG_WINDOW,
                                       min_periods=VOL_LONG_WINDOW).std()

    # ── Trend ─────────────────────────────────────────────────────────────────
    # ma_gap_8/26: Friday close relative to its N-week moving average
    df["ma_gap_8"]  = (df["close"]
                       / df["close"].rolling(MA_SHORT_WINDOW,
                                             min_periods=MA_SHORT_WINDOW).mean()
                       - 1)
    df["ma_gap_26"] = (df["close"]
                       / df["close"].rolling(MA_LONG_WINDOW,
                                             min_periods=MA_LONG_WINDOW).mean()
                       - 1)

    # drawdown_52w: Friday close relative to 52-week rolling maximum
    # rolling(52).max() at t = max(close[t-51 : t+1]), uses data <= t
    df["drawdown_52w"] = (df["close"]
                          / df["close"].rolling(DRAWDOWN_WINDOW,
                                                min_periods=DRAWDOWN_WINDOW).max()
                          - 1)

    # ── Volume / Liquidity ─────────────────────────────────────────────────────
    # vol_chg_4w: current-week dollar vol relative to 4 weeks ago
    # shift(4) gives dollar_vol at t-4 (strictly past data)
    df["vol_chg_4w"] = df["dollar_vol"] / df["dollar_vol"].shift(VOL_CHG_WINDOW) - 1

    # adv_12w_log: log(1 + 12-week average weekly dollar volume)
    df["adv_12w_log"] = np.log1p(
        df["dollar_vol"].rolling(ADV_WINDOW, min_periods=ADV_WINDOW).mean()
    )

    return df

print("_compute_ticker_features() defined.")

The feature function is applied across all ticker groups. Sanity checks confirm the absence of infinite values, duplicate rows, and any feature that uses `shift(-1)`.

In [ ]:
# ── Apply per-ticker feature computation ─────────────────────────────────────
print("Computing features across all tickers (groupby apply)...")
_t0 = time.time()

panel = (panel
         .sort_values(["ticker", "date"])
         .groupby("ticker", group_keys=False)
         .apply(_compute_ticker_features)
         .reset_index(drop=True))

print(f"Feature engineering complete in {time.time() - _t0:.1f}s.")
print(f"Panel shape after feature engineering: {panel.shape}")

# ── Sanity check 1: no infinite values ───────────────────────────────────────
_feature_cols_so_far = [
    "ret_t", "mom_4w", "mom_12w",
    "vol_8w", "vol_26w",
    "ma_gap_8", "ma_gap_26", "drawdown_52w",
    "vol_chg_4w", "adv_12w_log",
    "spy_ret_t",
]

_inf_counts = {c: np.isinf(panel[c]).sum() for c in _feature_cols_so_far}
_has_inf = {c: v for c, v in _inf_counts.items() if v > 0}
if _has_inf:
    print(f"WARNING: infinite values found: {_has_inf}")
    # Replace inf with NaN so they are treated as missing
    panel.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("  → replaced with NaN.")
else:
    print("Sanity check 1 PASSED: no infinite values in features.")

# ── Sanity check 2: no duplicate (date, ticker) rows ─────────────────────────
_n_dupes = panel.duplicated(subset=["date", "ticker"]).sum()
assert _n_dupes == 0, f"Duplicate (date, ticker) rows found: {_n_dupes}"
print(f"Sanity check 2 PASSED: no duplicate (date, ticker) rows.")

# ── Sanity check 3: dates are ascending within each ticker ───────────────────
_bad_order = (panel.groupby("ticker")["date"]
              .apply(lambda x: (x.diff().dropna() < pd.Timedelta(0)).any()))
assert not _bad_order.any(), f"Date ordering violated for: {_bad_order[_bad_order].index.tolist()}"
print(f"Sanity check 3 PASSED: dates are ascending within each ticker.")

# ── Sanity check 4: report NaN rates from rolling burn-in ────────────────────
_nan_rates = panel[_feature_cols_so_far].isna().mean().round(4)
print(f"\nNaN rates per feature (from rolling burn-in, expected ~52/N_dates):")
for feat, rate in _nan_rates.items():
    print(f"  {feat:20s}: {rate:.2%}")

# ── Confirm: no feature uses shift(-1) ───────────────────────────────────────
# Manual verification: _compute_ticker_features() uses only:
#   .rolling(N).sum() / .std() / .mean() / .max()  — backward-looking ✓
#   .shift(4)  for vol_chg_4w lag   — uses data 4 periods BEFORE t  ✓
#   No .shift(-1) appears in any feature column.
#
# The ONLY use of shift(-1) in this notebook is in Section 7 (ret_fwd1 target construction).
print("Timing check: no feature uses shift(-1).  [confirmed by code inspection]")
print("shift(-1) is reserved exclusively for ret_fwd1 / target_up in Section 7.")

## 6) Feature Table

The table below documents all 11 features used in the ML signal, including each feature's timing rule and a note confirming the absence of look-ahead leakage.

`FEATURE_COLS` is defined as the official ordered feature list. The `feature_dictionary_table` documents each feature's name, category, timing rule, and leakage note.

In [ ]:
# ── Official feature column list ─────────────────────────────────────────────
# This list is used in all subsequent modelling, walk-forward, and audit steps.
FEATURE_COLS: list = [
    "ret_t",          # Price/Momentum — current weekly return
    "mom_4w",         # Price/Momentum — 4-week cumulative return
    "mom_12w",        # Price/Momentum — 12-week cumulative return
    "vol_8w",         # Risk            — 8-week realised weekly vol
    "vol_26w",        # Risk            — 26-week realised weekly vol
    "ma_gap_8",       # Trend           — close vs 8-week MA
    "ma_gap_26",      # Trend           — close vs 26-week MA
    "drawdown_52w",   # Trend           — close vs 52-week high
    "vol_chg_4w",     # Volume/Liquidity — 4-week volume change
    "adv_12w_log",    # Volume/Liquidity — log 12-week avg dollar volume
    "spy_ret_t",      # Market State    — SPY weekly return
]

assert all(c in panel.columns for c in FEATURE_COLS), "Missing feature columns in panel"
print(f"FEATURE_COLS defined: {len(FEATURE_COLS)} features")

_feature_dict_data = [
    # (feature, category, definition, timing, leakage_note)
    ("ret_t",       "Price/Momentum",
     "Current weekly return: (close_t - close_{t-1}) / close_{t-1}",
     "Available at Friday close t",
     "No leakage: uses close prices at t and t-1 only"),

    ("mom_4w",      "Price/Momentum",
     "4-week cumulative return: sum(ret[t-3 : t])",
     "Uses weekly returns dated ≤ t",
     "No leakage: backward rolling sum including t"),

    ("mom_12w",     "Price/Momentum",
     "12-week cumulative return: sum(ret[t-11 : t])",
     "Uses weekly returns dated ≤ t",
     "No leakage: backward rolling sum including t"),

    ("vol_8w",      "Risk",
     "8-week realised weekly return std: std(ret[t-7 : t])",
     "Uses weekly returns dated ≤ t",
     "No leakage: backward rolling std including t"),

    ("vol_26w",     "Risk",
     "26-week realised weekly return std: std(ret[t-25 : t])",
     "Uses weekly returns dated ≤ t",
     "No leakage: backward rolling std including t"),

    ("ma_gap_8",    "Trend",
     "Close relative to 8-week moving average: close_t / MA8_t - 1",
     "Uses close prices dated ≤ t",
     "No leakage: MA computed on close prices through t"),

    ("ma_gap_26",   "Trend",
     "Close relative to 26-week moving average: close_t / MA26_t - 1",
     "Uses close prices dated ≤ t",
     "No leakage: MA computed on close prices through t"),

    ("drawdown_52w","Trend",
     "Distance from 52-week high: close_t / max(close[t-51 : t]) - 1",
     "Uses close prices dated ≤ t",
     "No leakage: rolling max includes only prices through t"),

    ("vol_chg_4w",  "Volume/Liquidity",
     "Weekly dollar volume change: dollar_vol_t / dollar_vol_{t-4} - 1",
     "Uses weekly dollar volume at t and t-4",
     "No leakage: t-4 is strictly past data; shift(4) used"),

    ("adv_12w_log", "Volume/Liquidity",
     "Log(1 + 12-week average weekly dollar volume)",
     "Uses weekly dollar volumes dated ≤ t",
     "No leakage: backward rolling mean including t"),

    ("spy_ret_t",   "Market State",
     "SPY weekly return at t — market-wide factor control",
     "Available at Friday close t (same as all other features)",
     "No leakage: SPY return at t is observable before portfolio formation"),
]

feature_dictionary_table = pd.DataFrame(
    _feature_dict_data,
    columns=["feature", "category", "definition", "timing", "leakage_note"]
)

save_table(feature_dictionary_table, "feature_dictionary_table")
show_table(feature_dictionary_table[["feature","category","timing","leakage_note"]])

## 7) Missingness Handling

The target variable `ret_fwd1` is constructed with `shift(-1)` **within ticker groups** — the only use of `shift(-1)` in this notebook. Terminal rows (no forward return) are then dropped. The `missingness_audit_table` records every cleaning step with row counts.

The forward return `ret_fwd1` is constructed as `shift(-1)` within ticker groups. This is the only use of `shift(-1)` in the notebook. The direction label `target_up = 1(ret_fwd1 > 0)` is the classification target.

In [ ]:
# ── Step 1: compute ret_fwd1 and target_up using shift(-1) ──────────────────
# IMPORTANT: shift(-1) is used here only, and only within ticker groups.
# This is the legitimate target construction; it is NOT a feature.

def _add_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add ret_fwd1 and target_up to a single-ticker time series.
    shift(-1) aligns next week's return with the current decision date.
    """
    df = df.sort_values("date").copy()
    df["ret_fwd1"]   = df["ret"].shift(-1)                       # ONLY shift(-1) in notebook
    df["target_up"]  = np.where(                                  # NaN preserved for terminal row
        df["ret_fwd1"].isna(), np.nan,
        (df["ret_fwd1"] > 0).astype(int)
    )
    return df

panel = (panel
         .groupby("ticker", group_keys=False)
         .apply(_add_target)
         .reset_index(drop=True))

_n_before_target_drop = len(panel)
_n_nan_target = panel["ret_fwd1"].isna().sum()
print(f"Rows before dropping terminal rows: {_n_before_target_drop:,}")
print(f"Rows with NaN ret_fwd1 (terminal)  : {_n_nan_target:,}")
print(f"  (expected: {len(FINAL_TICKERS)} — one per ticker)")

Rows affected by rolling feature burn-in and terminal missing returns are dropped. The 6-step `missingness_audit_table` records every cleaning action with row counts.

In [ ]:
# ── Step 2: count and drop feature burn-in rows (NaN in FEATURE_COLS) ────────
_nan_feature_mask   = panel[FEATURE_COLS].isna().any(axis=1)
_nan_target_mask    = panel["ret_fwd1"].isna()

# burn-in: NaN in features (regardless of target)
_n_burnin_dropped   = int(_nan_feature_mask.sum())

# terminal: NaN in target but NOT in features
_n_terminal_dropped = int((_nan_target_mask & ~_nan_feature_mask).sum())

# ── Step 3: drop all rows with any NaN in FEATURE_COLS or target ──────────────
_drop_mask = _nan_feature_mask | _nan_target_mask
panel = panel[~_drop_mask].reset_index(drop=True)
_n_final = len(panel)

# Convert target_up to int (NaN rows already dropped above)
panel["target_up"] = panel["target_up"].astype(int)

print(f"Burn-in rows dropped  (step 4): {_n_burnin_dropped:,}")
print(f"Terminal rows dropped (step 5): {_n_terminal_dropped:,}")
print(f"Final modelling rows  (step 6): {_n_final:,}")
print(f"Final unique dates            : {panel['date'].nunique():,}")
print(f"Final unique tickers          : {panel['ticker'].nunique():,}")

# ── Build full missingness_audit_table (6 steps) ─────────────────────────────
_rows_ticker_removed = _n_rows_raw - (_n_weekly_dates * len(FINAL_TICKERS))
_rows_price_dropped  = (_n_weekly_dates * len(FINAL_TICKERS)) - _n_rows_after_price

missingness_audit_table = pd.DataFrame([
    {
        "step"       : 1,
        "description": "Raw download (theoretical maximum)",
        "rows_removed": 0,
        "rows_remaining": _n_rows_raw,
        "reason"     : (f"All {_n_weekly_dates} weekly dates × "
                        f"{_n_tickers_attempted} attempted tickers"),
    },
    {
        "step"       : 2,
        "description": "Tickers removed (failed / insufficient history)",
        "rows_removed": _rows_ticker_removed,
        "rows_remaining": _n_weekly_dates * len(FINAL_TICKERS),
        "reason"     : (f"{len(FAILED_TICKERS)} tickers removed "
                        f"({', '.join(FAILED_TICKERS) if FAILED_TICKERS else 'none'})"),
    },
    {
        "step"       : 3,
        "description": "Missing close / return / dollar volume rows",
        "rows_removed": _rows_price_dropped,
        "rows_remaining": _n_rows_after_price,
        "reason"     : ("First pct_change NaN per ticker + any trading halts "
                        "or data gaps in yfinance"),
    },
    {
        "step"       : 4,
        "description": "Rolling feature burn-in (NaN in FEATURE_COLS)",
        "rows_removed": _n_burnin_dropped,
        "rows_remaining": _n_rows_after_price - _n_burnin_dropped,
        "reason"     : (f"Longest rolling window: drawdown_52w = {DRAWDOWN_WINDOW} weeks; "
                        "earlier rows have insufficient lookback history"),
    },
    {
        "step"       : 5,
        "description": "Terminal rows (no forward return)",
        "rows_removed": _n_terminal_dropped,
        "rows_remaining": _n_final,
        "reason"     : (f"Last row per ticker has no ret_fwd1; "
                        f"exactly {len(FINAL_TICKERS)} rows removed "
                        f"(one per final ticker)"),
    },
    {
        "step"       : 6,
        "description": "Final modelling rows",
        "rows_removed": 0,
        "rows_remaining": _n_final,
        "reason"     : "Clean panel used for all walk-forward modelling",
    },
])

save_table(missingness_audit_table, "missingness_audit_table")
show_table(missingness_audit_table)

universe_summary_table = pd.DataFrame([
    {"item": "Initial tickers attempted",
     "value": str(_n_tickers_attempted)},
    {"item": "Tickers removed (failed / insufficient data)",
     "value": str(len(FAILED_TICKERS))},
    {"item": "Final tickers (FINAL_TICKERS)",
     "value": str(len(FINAL_TICKERS))},
    {"item": "Number of sectors",
     "value": str(sector_count_table["sector"].nunique())},
    {"item": "Weekly Friday-close cross-section size",
     "value": str(len(FINAL_TICKERS))},
    {"item": "Total weekly dates (full calendar)",
     "value": str(_n_weekly_dates)},
    {"item": "Final modelling rows",
     "value": f"{_n_final:,}"},
    {"item": "Modelling sample start",
     "value": str(panel["date"].min().date())},
    {"item": "Modelling sample end",
     "value": str(panel["date"].max().date())},
    {"item": "Unique decision dates",
     "value": str(panel["date"].nunique())},
])

save_table(universe_summary_table, "universe_summary_table")
show_table(universe_summary_table)

print(f"Clean panel: {len(panel):,} rows, {panel['ticker'].nunique()} tickers, "
      f"{panel['date'].nunique()} dates, "
      f"positive-class rate {panel['target_up'].mean():.4f}")

## 8) Leakage Prevention Statement

The following rules are enforced throughout this notebook to prevent look-ahead bias:

| Rule | Implementation |
|------|----------------|
| Features use data dated ≤ t | All rolling windows are backward-looking; no `center=True` |
| `shift(-1)` reserved for target only | Used only in `_add_target()` to construct `ret_fwd1` / `target_up` |
| Train data excludes current date | Walk-forward trains on `date < decision_date` |
| StandardScaler fitted inside Pipeline | Never fitted on test or full sample |
| Hyperparameters frozen before holdout | Selected on pre-holdout validation block; not re-tuned on official OOS data |
| Vol targeting uses `.shift(1)` | Trailing vol estimate excludes the current week |

Formal verification of all 18 checks is in section 19 (Leakage Audit).

## 9) Model Definitions

Two models are compared under the same walk-forward backtest design:

| Model | Type | Key hyperparameters |
|-------|------|--------------------|
| Logistic Regression | Linear baseline | C tuned over [0.3, 1.0, 3.0] |
| Random Forest | Tree-based | max_depth × min_samples_leaf grid; n_estimators = 300 |

Hyperparameters are selected **once** on a 26-week time-ordered validation block inside the pre-holdout sample. The selection metric is validation net Sharpe at 5 bps, using the same portfolio pipeline as the official backtest. Parameters are frozen before the official holdout is examined.

**This is not K-fold CV and not random CV.** The official holdout is never used in any tuning or selection step.

In [ ]:
# ── Helper: raw weights for a single date ────────────────────────────────────
def _make_raw_weights_local(
    scores: "pd.Series",
    q: float,
    all_tickers: list,
) -> "pd.Series":
    """
    Top q% long (+1/k), bottom q% short (-1/k), equal-weight.
    Returns full all_tickers vector (missing tickers = 0).
    Local helper used only in Section 9 validation — identical logic to Section 11.
    """
    import pandas as _pd, numpy as _np
    w = _pd.Series(0.0, index=all_tickers)
    available = scores.reindex(all_tickers).dropna()
    k = max(1, int(_np.floor(len(all_tickers) * q)))
    w[available.nlargest(k).index]  = +1.0 / k
    w[available.nsmallest(k).index] = -1.0 / k
    return w


# ── Helper: drifted weights (local copy for use in Section 9) ───────────────────
def _compute_drifted_weights_local(
    w_prev: "pd.Series",
    asset_rets: "pd.Series",
) -> "pd.Series":
    """
    Drift the previous scaled weight vector through realized asset returns.

    w⁺_{t-1} = w_{t-1} × (1 + r_{asset,t-1}) / (1 + r_{portfolio,t-1})

    This is the weight vector AFTER returns are realized but BEFORE
    rebalancing at t. Used to compute one-way drifted turnover.

    Parameters
    ----------
    w_prev     : Scaled weights from the previous period.
    asset_rets : Realized returns for the same previous period.

    Returns
    -------
    Drifted weight vector (same index as w_prev).
    """
    import numpy as _np
    port_ret = float((w_prev * asset_rets).sum())
    denom    = 1.0 + port_ret
    if abs(denom) < 1e-12:          # degenerate: return unchanged
        return w_prev.copy()
    return w_prev * (1.0 + asset_rets) / denom


# ── Helper: validation net Sharpe with past-only vol targeting ────────────────
def _val_net_sharpe(
    model,
    train_df_:  "pd.DataFrame",
    seed_df_:   "pd.DataFrame",
    val_df_:    "pd.DataFrame",
) -> float:
    """
    Fit model on train_df_ and compute validation net Sharpe @5 bps.

    Transaction costs use DRIFTED one-way turnover (same formula as official
    backtest) to ensure the selection metric matches the final objective.

    Volatility targeting uses past-only raw gross returns at every step:
    - Seeded from seed_df_ (last VOL_WINDOW training dates) using in-sample
      model scores. These seed gross returns are used for vol initialisation
      only and do not affect which dates are evaluated out-of-sample.
    - Updated with the current week's raw gross return only after that week's
      scaler and return have been computed, so the current week does not
      affect its own scaler.

    No validation or holdout data enters the vol estimate before its own date.

    Parameters
    ----------
    model     : Unfitted sklearn estimator.
    train_df_ : Training rows (date < VALIDATION_START).
    seed_df_  : Last VOL_WINDOW rows from training (vol initialisation).
    val_df_   : Validation rows (26 weeks).

    Returns
    -------
    Annualised net Sharpe (float); -999 if insufficient data.
    """
    import numpy as _np

    # ── 1. Fit model on full training set ────────────────────────────────────
    model.fit(train_df_[FEATURE_COLS], train_df_["target_up"])
    target_wvol = VOL_TARGET_ANN / _np.sqrt(PERIODS_PER_YEAR)

    # ── 2. Seed raw gross returns from last VOL_WINDOW training dates ────────
    past_gross = []
    for dt in sorted(seed_df_["date"].unique()):
        day = seed_df_[seed_df_["date"] == dt]
        if day.empty:
            continue
        proba  = model.predict_proba(day[FEATURE_COLS])[:, 1]
        scores = pd.Series(proba, index=day["ticker"].values)
        raw_w  = _make_raw_weights_local(scores, BASE_QUANTILE, FINAL_TICKERS)
        r_vec  = (day.set_index("ticker")["ret_fwd1"]
                  .reindex(FINAL_TICKERS).fillna(0.0))
        past_gross.append(float((raw_w * r_vec).sum()))

    # ── 3. Validation loop with drifted turnover ─────────────────────────────
    net_rets  = []
    prev_sw   = None   # previous SCALED weight vector (for drifted turnover)
    prev_r_vec = None  # asset returns from previous week (for drift calc)

    val_dates = sorted(val_df_["date"].unique())
    for dt in val_dates:
        day = val_df_[val_df_["date"] == dt]
        if day.empty:
            continue

        proba  = model.predict_proba(day[FEATURE_COLS])[:, 1]
        scores = pd.Series(proba, index=day["ticker"].values)
        raw_w  = _make_raw_weights_local(scores, BASE_QUANTILE, FINAL_TICKERS)

        # Asset returns for this week (used for drifted turnover next week)
        r_vec = (day.set_index("ticker")["ret_fwd1"]
                 .reindex(FINAL_TICKERS).fillna(0.0))

        raw_g = float((raw_w * r_vec).sum())   # raw gross return this week

        # ── Past-only trailing vol (seeded, then updated week-by-week) ───────
        n_past = len(past_gross)
        if n_past >= VOL_WINDOW:
            trail_std = float(_np.std(past_gross[-VOL_WINDOW:], ddof=1))
        elif n_past > 1:
            trail_std = float(_np.std(past_gross, ddof=1))
        else:
            trail_std = target_wvol      # default → scaler = 1

        scaler   = (float(_np.clip(target_wvol / trail_std, *SCALER_CLIP))
                    if trail_std > 1e-10 else 1.0)
        scaled_w = raw_w * scaler

        gross_ret = float((scaled_w * r_vec).sum())

        # ── Drifted one-way turnover ─────────────────────────────────────────
        if prev_sw is None:
            # First validation week: no prior drifted weight available
            turnover = 0.5 * float(scaled_w.abs().sum())
        else:
            # Drift previous scaled weight through last week's asset returns
            drifted  = _compute_drifted_weights_local(prev_sw, prev_r_vec)
            turnover = 0.5 * float((scaled_w - drifted).abs().sum())

        net_rets.append(gross_ret - turnover * BASE_COST_BPS / 10_000)

        # Update state for next iteration
        past_gross.append(raw_g)
        prev_sw    = scaled_w
        prev_r_vec = r_vec

    if len(net_rets) < 4:
        return -999.0
    a   = _np.array(net_rets)
    std = float(_np.std(a, ddof=1))
    return float(_np.mean(a) / std * _np.sqrt(PERIODS_PER_YEAR)) if std > 1e-10 else 0.0


print("Helper functions defined:")
print("  _make_raw_weights_local()")
print("  _compute_drifted_weights_local()")
print("  _val_net_sharpe()  [uses drifted one-way turnover]")

## 10) Backtest Helper Functions

Production functions used throughout the official backtest are defined here: performance metrics, portfolio construction, volatility targeting, turnover, and the backtest runner.

Performance metric functions — annualised return, volatility, Sharpe ratio, maximum drawdown, and cumulative wealth — are standardised here for consistent use across tearsheets, benchmarks, and robustness checks.

In [ ]:
# ── Final production performance functions ───────────────────────────────────
# All use PERIODS_PER_YEAR = 52 (weekly frequency).

def annualized_return(r: pd.Series,
                      periods: int = PERIODS_PER_YEAR) -> float:
    """Mean weekly return × periods per year."""
    return float(r.mean() * periods)


def annualized_volatility(r: pd.Series,
                           periods: int = PERIODS_PER_YEAR) -> float:
    """Weekly return std × sqrt(periods per year)."""
    return float(r.std(ddof=1) * np.sqrt(periods))


def sharpe_ratio(r: pd.Series,
                 periods: int = PERIODS_PER_YEAR) -> float:
    """Annualised return / annualised volatility (no risk-free rate subtracted)."""
    v = annualized_volatility(r, periods)
    return float(annualized_return(r, periods) / v) if v > 1e-10 else 0.0


def cumulative_wealth(r: pd.Series) -> pd.Series:
    """Cumulative wealth index starting at 1.0."""
    return (1 + r).cumprod()


def drawdown_series(r: pd.Series) -> pd.Series:
    """Drawdown from running maximum of cumulative wealth."""
    cw = cumulative_wealth(r)
    return cw / cw.cummax() - 1


def max_drawdown(r: pd.Series) -> float:
    """Maximum drawdown (negative number)."""
    return float(drawdown_series(r).min())


def make_tearsheet(
    gross_r:    pd.Series,
    net_r:      pd.Series,
    turnover_s: pd.Series,
    model_name: str,
) -> dict:
    """
    Compute full tear sheet metrics for one model.

    Returns a dict with gross and net performance plus average turnover.
    All metrics use scaled (official) strategy returns.
    """
    return {
        "model"             : model_name,
        "gross_ann_ret"     : round(annualized_return(gross_r),    4),
        "gross_ann_vol"     : round(annualized_volatility(gross_r),4),
        "gross_sharpe"      : round(sharpe_ratio(gross_r),         4),
        "net_ann_ret_5bps"  : round(annualized_return(net_r),      4),
        "net_ann_vol"       : round(annualized_volatility(net_r),  4),
        "net_sharpe_5bps"   : round(sharpe_ratio(net_r),           4),
        "net_max_drawdown"  : round(max_drawdown(net_r),           4),
        "avg_turnover"      : round(float(turnover_s.mean()),       4),
    }


print("Performance functions defined:")
for fn in ["annualized_return", "annualized_volatility", "sharpe_ratio",
           "cumulative_wealth", "drawdown_series", "max_drawdown", "make_tearsheet"]:
    print(f"  {fn}()")

Portfolio construction functions convert ranked predicted-probability scores into a long/short weight vector over the full 45-ticker universe.

In [ ]:
# ── Portfolio construction functions ─────────────────────────────────────────

def make_portfolio_weights(
    scores: pd.Series,
    q: float = BASE_QUANTILE,
    all_tickers: list = None,
) -> pd.Series:
    """
    Convert predicted-probability scores to a long-short weight vector.

    Top q% by score → long (+1/k each).
    Bottom q% by score → short (−1/k each).
    All other tickers → weight 0.
    Returns a full vector over all_tickers (default: FINAL_TICKERS).

    Parameters
    ----------
    scores      : pd.Series mapping ticker → predicted probability.
    q           : Quantile cutoff (default BASE_QUANTILE = 0.20).
    all_tickers : Complete ticker universe (default FINAL_TICKERS).

    Returns
    -------
    pd.Series of weights indexed by all_tickers.
    """
    if all_tickers is None:
        all_tickers = FINAL_TICKERS

    k = max(1, int(np.floor(len(all_tickers) * q)))
    available = scores.reindex(all_tickers).dropna()

    w = pd.Series(0.0, index=all_tickers, dtype=float)
    w[available.nlargest(k).index]  = +1.0 / k   # long leg
    w[available.nsmallest(k).index] = -1.0 / k   # short leg
    return w


def make_weight_panel_from_scores(
    score_panel: pd.DataFrame,
    q: float = BASE_QUANTILE,
) -> pd.DataFrame:
    """
    Apply make_portfolio_weights to every row of a score panel.

    Parameters
    ----------
    score_panel : DataFrame of shape (T, N) with tickers as columns.
    q           : Quantile cutoff.

    Returns
    -------
    DataFrame of weights, same shape as score_panel.
    """
    weights = score_panel.apply(
        lambda row: make_portfolio_weights(row, q=q),
        axis=1,
    )
    return weights


print("make_portfolio_weights()        defined.")
print("make_weight_panel_from_scores() defined.")
print(f"Base quantile: q = {BASE_QUANTILE}")
print(f"  k = floor({len(FINAL_TICKERS)} × {BASE_QUANTILE}) = "
      f"{int(np.floor(len(FINAL_TICKERS) * BASE_QUANTILE))} names per leg")

The volatility-targeting function scales raw portfolio weights to target 10% annualised volatility using a trailing 26-week realised-volatility estimator.

In [ ]:
# ── Formal apply_vol_targeting() ─────────────────────────────────────────────

def apply_vol_targeting(
    raw_weights_df: pd.DataFrame,
    asset_returns_df: pd.DataFrame,
    target_ann_vol: float = VOL_TARGET_ANN,
    window: int = VOL_WINDOW,
    clip: tuple = SCALER_CLIP,
    periods_per_year: int = PERIODS_PER_YEAR,
) -> tuple:
    """
    Apply past-only volatility targeting to raw portfolio weights.

    Steps:
    1. Compute raw gross return series from raw_weights × asset_returns.
    2. Estimate trailing weekly vol: rolling(window).std().shift(1).
       .shift(1) guarantees week t's return does NOT affect its own scaler.
       Rows where trailing vol is NaN (first `window` weeks) get scaler = 1.0.
    3. Compute scaler = target_weekly_vol / trailing_vol, clipped to `clip`.
    4. Scale raw_weights by the scaler at each date.

    Parameters
    ----------
    raw_weights_df   : DataFrame (T × N) of raw equal-weight L/S weights.
    asset_returns_df : DataFrame (T × N) of realized weekly returns per ticker,
                       aligned to the same dates as raw_weights_df.
                       (These are ret_fwd1 for the holding period t+1.)
    target_ann_vol   : Annualised vol target (default VOL_TARGET_ANN = 0.10).
    window           : Rolling window for trailing vol (default VOL_WINDOW = 26).
    clip             : (min, max) scaler clip bounds (default SCALER_CLIP).
    periods_per_year : Annualisation factor (default PERIODS_PER_YEAR = 52).

    Returns
    -------
    scaled_weights : DataFrame (T × N) — OFFICIAL base strategy weights.
    scaler_series  : Series (T,)  — scaler applied at each date.
    raw_gross_returns : Series (T,) — raw gross return series (diagnostic).
    """
    target_weekly_vol = target_ann_vol / np.sqrt(periods_per_year)

    # ── Step 1: raw gross return series ──────────────────────────────────────
    # Align asset returns to weight dates; fill missing tickers with 0.
    _rets = asset_returns_df.reindex(
        index=raw_weights_df.index,
        columns=raw_weights_df.columns,
    ).fillna(0.0)

    raw_gross_returns = (raw_weights_df * _rets).sum(axis=1)

    # ── Step 2: trailing weekly vol (past-only via .shift(1)) ────────────────
    # .shift(1) means: at date t, the vol estimate uses returns up to t-1 only.
    trailing_vol = (raw_gross_returns
                    .rolling(window=window, min_periods=window)
                    .std()
                    .shift(1))          # <── critical: no current-week data

    # ── Step 3: scaler ────────────────────────────────────────────────────────
    # Where trailing_vol is NaN (first `window` weeks after shift), use 1.0.
    scaler_series = np.where(
        trailing_vol.isna() | (trailing_vol < 1e-10),
        1.0,                                            # default before history
        np.clip(target_weekly_vol / trailing_vol, *clip),
    )
    scaler_series = pd.Series(scaler_series,
                              index=raw_weights_df.index,
                              name="scaler")

    # ── Step 4: scale weights ─────────────────────────────────────────────────
    scaled_weights = raw_weights_df.multiply(scaler_series, axis=0)

    return scaled_weights, scaler_series, raw_gross_returns


print("apply_vol_targeting() defined.")
print(f"  target annual vol : {VOL_TARGET_ANN*100:.0f}%")
print(f"  target weekly vol : {VOL_TARGET_ANN/np.sqrt(PERIODS_PER_YEAR)*100:.4f}%")
print(f"  trailing window   : {VOL_WINDOW} weeks")
print(f"  scaler clip       : {SCALER_CLIP}")
print(f"  .shift(1) applied : current week excluded from its own scaler ✓")

The drifted one-way turnover function and the official backtest runner are defined here. The backtest runner computes gross and net returns from scaled weights using the full asset vector.

In [ ]:
# ── Formal compute_drifted_weights() ─────────────────────────────────────────

def compute_drifted_weights(
    w_prev: pd.Series,
    asset_rets: pd.Series,
) -> pd.Series:
    """
    Drift the previous period's weight vector through realized asset returns.

    w⁺_{t-1} = w_{t-1} × (1 + r_{asset,t-1}) / (1 + r_{portfolio,t-1})

    This is the weight vector AFTER returns are realized but BEFORE
    rebalancing at t. It is used to compute drifted one-way turnover.

    Parameters
    ----------
    w_prev     : Scaled weights from the previous period (pd.Series over FINAL_TICKERS).
    asset_rets : Realized asset returns for the PREVIOUS holding period (ret_fwd1 at t-1).
                 Must be aligned to the same index as w_prev.

    Returns
    -------
    Drifted weight vector (pd.Series, same index as w_prev).
    """
    asset_rets_aligned = asset_rets.reindex(w_prev.index).fillna(0.0)
    port_ret = float((w_prev * asset_rets_aligned).sum())
    denom    = 1.0 + port_ret
    if abs(denom) < 1e-12:
        return w_prev.copy()      # degenerate: portfolio wiped out
    return w_prev * (1.0 + asset_rets_aligned) / denom


def compute_one_way_turnover(
    w_curr: pd.Series,
    w_drifted: pd.Series,
) -> float:
    """
    Compute one-way portfolio turnover between current and drifted prior weights.

    turnover_t = 0.5 × Σ_i |w_{i,t} − w⁺_{i,t-1}|

    Both vectors must span the full FINAL_TICKERS universe (including zeros).
    Excluding zeros would understate turnover when names enter/exit the portfolio.

    Parameters
    ----------
    w_curr    : Current target weights (scaled, full vector).
    w_drifted : Drifted prior weights (full vector, after returns, before rebalance).

    Returns
    -------
    One-way turnover (float, non-negative).
    """
    return 0.5 * float((w_curr - w_drifted).abs().sum())


print("compute_drifted_weights()  defined.")
print("compute_one_way_turnover() defined.")

# ── Formal run_official_backtest_from_weights() ──────────────────────────────

def run_official_backtest_from_weights(
    scaled_weights_df: pd.DataFrame,
    asset_returns_df: pd.DataFrame,
    cost_bps: float = BASE_COST_BPS,
    label: str = "",
) -> dict:
    """
    Run the official OOS backtest from a scaled weight panel.

    Uses drifted one-way turnover over the full asset vector.
    Net return = gross return − turnover × cost_bps / 10 000.

    Parameters
    ----------
    scaled_weights_df : DataFrame (T × N) — OFFICIAL scaled weights from Section 12.
    asset_returns_df  : DataFrame (T × N) — realized ret_fwd1 per ticker per date.
    cost_bps          : One-way transaction cost in basis points.
    label             : String label for print output (e.g. 'LR', 'RF').

    Returns
    -------
    dict with keys:
        gross_returns  : pd.Series (T,)
        net_returns    : pd.Series (T,)
        turnover       : pd.Series (T,)
        cost_bps       : float (cost assumption used)
    """
    # ── Enforce full FINAL_TICKERS vector ──────────────────────────────────
    scaled_weights_df = scaled_weights_df.reindex(
        columns=FINAL_TICKERS).fillna(0.0)
    asset_returns_df  = asset_returns_df.reindex(
        columns=FINAL_TICKERS).fillna(0.0)
    assert scaled_weights_df.shape[1] == len(FINAL_TICKERS), (
        f"scaled_weights_df columns ({scaled_weights_df.shape[1]}) "
        f"!= len(FINAL_TICKERS) ({len(FINAL_TICKERS)})")
    assert asset_returns_df.shape[1]  == len(FINAL_TICKERS), (
        f"asset_returns_df columns ({asset_returns_df.shape[1]}) "
        f"!= len(FINAL_TICKERS) ({len(FINAL_TICKERS)})")

    dates = scaled_weights_df.index.tolist()
    N     = len(dates)

    gross_list = []
    net_list   = []
    to_list    = []

    _prev_w    = None
    _prev_rets = None

    for _j, dt in enumerate(dates):
        w_curr = scaled_weights_df.loc[dt].reindex(FINAL_TICKERS).fillna(0.0)
        r_curr = (asset_returns_df
                  .loc[dt]
                  .reindex(FINAL_TICKERS)
                  .fillna(0.0))

        # ── Gross return ──────────────────────────────────────────────────────
        gross = float((w_curr * r_curr).sum())

        # ── Drifted one-way turnover ──────────────────────────────────────────
        if _prev_w is None:
            # First OOS week: no prior holdings
            turnover = 0.5 * float(w_curr.abs().sum())
        else:
            # Drift previous weights through PREVIOUS period's asset returns
            w_drifted = compute_drifted_weights(_prev_w, _prev_rets)
            turnover  = compute_one_way_turnover(w_curr, w_drifted)

        # ── Net return ────────────────────────────────────────────────────────
        net = gross - turnover * cost_bps / 10_000

        gross_list.append(gross)
        net_list.append(net)
        to_list.append(turnover)

        # Update state: store current weights and current realized returns
        # (used to drift weights when computing NEXT week's turnover)
        _prev_w    = w_curr
        _prev_rets = r_curr      # ret_fwd1 at dt = return realized this period

    gross_s = pd.Series(gross_list, index=dates, name="gross_return")
    net_s   = pd.Series(net_list,   index=dates, name="net_return")
    to_s    = pd.Series(to_list,    index=dates, name="turnover")

    if label:
        print(f"{label}  gross mean={gross_s.mean():.5f}  "
              f"net mean={net_s.mean():.5f}  "
              f"avg turnover={to_s.mean():.4f}")

    return {
        "gross_returns" : gross_s,
        "net_returns"   : net_s,
        "turnover"      : to_s,
        "cost_bps"      : cost_bps,
    }


print("run_official_backtest_from_weights() defined.")

## 11) Walk-Forward Backtest

The modelling sample is split by date into three non-overlapping periods: training, validation, and test (official out-of-sample holdout). The training period is used to estimate the models, the validation period is used once for pre-test hyperparameter selection, and the final 104 weekly dates are reserved as the test period for official out-of-sample performance evaluation.

The test period is the official out-of-sample holdout. It is not used for hyperparameter selection or model-design decisions.

All rows from the same date are assigned to the same split. This date-based split avoids panel leakage because stocks observed in the same week share the same market information.

The sample is partitioned into three non-overlapping date sets based on date alone — not row index. During the official walk-forward evaluation, the training set expands at each monthly refit, while the test period (official out-of-sample holdout) is never used for tuning.

In [ ]:
# ── Identify all unique weekly decision dates ─────────────────────────────────
all_dates = sorted(panel["date"].unique())
n_total_dates = len(all_dates)
print(f"Total unique decision dates in panel: {n_total_dates}")

# ── Define split boundaries ───────────────────────────────────────────────────
# Official holdout: last N_HOLDOUT_WEEKS dates
if n_total_dates < N_HOLDOUT_WEEKS + N_VALIDATION_WEEKS + 52:
    raise ValueError(
        f"Insufficient dates ({n_total_dates}) for holdout ({N_HOLDOUT_WEEKS}) + "
        f"validation ({N_VALIDATION_WEEKS}) + minimum training (52)."
    )

HOLDOUT_START    = all_dates[-N_HOLDOUT_WEEKS]
VALIDATION_START = all_dates[-(N_HOLDOUT_WEEKS + N_VALIDATION_WEEKS)]

holdout_dates    = [d for d in all_dates if d >= HOLDOUT_START]
validation_dates = [d for d in all_dates if VALIDATION_START <= d < HOLDOUT_START]
train_dates_list = [d for d in all_dates if d < VALIDATION_START]

# Convert to sets for fast membership testing
_holdout_set    = set(holdout_dates)
_validation_set = set(validation_dates)
_train_set      = set(train_dates_list)

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(holdout_dates)    == N_HOLDOUT_WEEKS,    (
    f"Expected {N_HOLDOUT_WEEKS} holdout dates, got {len(holdout_dates)}")
assert len(validation_dates) == N_VALIDATION_WEEKS, (
    f"Expected {N_VALIDATION_WEEKS} validation dates, got {len(validation_dates)}")
assert len(_holdout_set & _validation_set) == 0,    "Holdout and validation overlap!"
assert len(_holdout_set & _train_set)      == 0,    "Holdout and train overlap!"
assert len(_validation_set & _train_set)   == 0,    "Validation and train overlap!"
assert len(train_dates_list) >= 52,                 "Too few training dates!"

print(f"Official holdout start : {HOLDOUT_START.date()}")
print(f"Official holdout end   : {holdout_dates[-1].date()}")
print(f"Validation start       : {VALIDATION_START.date()}")
print(f"Validation end         : {validation_dates[-1].date()}")
print(f"Training dates         : {len(train_dates_list)}")
print(f"Validation dates       : {len(validation_dates)}")
print(f"Holdout dates          : {len(holdout_dates)}")

# ── Row subsets (used in tuning and audit) ────────────────────────────────────
train_df = panel[panel["date"].isin(_train_set)].copy()
val_df   = panel[panel["date"].isin(_validation_set)].copy()
hold_df  = panel[panel["date"].isin(_holdout_set)].copy()

print(f"train_df rows : {len(train_df):,}  "
      f"({train_df['date'].nunique()} dates × ~{len(FINAL_TICKERS)} tickers)")
print(f"val_df   rows : {len(val_df):,}  "
      f"({val_df['date'].nunique()} dates)")
print(f"hold_df  rows : {len(hold_df):,}  "
      f"({hold_df['date'].nunique()} dates)")

`train_holdout_date_table` records the exact start and end dates of each split (training / validation / test). `target_balance_table` reports the positive-class rate per split, confirming the label distribution is stable across periods.

In [ ]:
# ── train_holdout_date_table ──────────────────────────────────────────────────
train_holdout_date_table = pd.DataFrame([
    {
        "split"      : "Train",
        "start_date" : str(train_df["date"].min().date()),
        "end_date"   : str(train_df["date"].max().date()),
        "n_weeks"    : train_df["date"].nunique(),
        "n_rows"     : len(train_df),
        "note"       : "Model fitting; expands at each monthly refit in Section 10",
    },
    {
        "split"      : "Validation",
        "start_date" : str(val_df["date"].min().date()),
        "end_date"   : str(val_df["date"].max().date()),
        "n_weeks"    : val_df["date"].nunique(),
        "n_rows"     : len(val_df),
        "note"       : "One-time time-ordered block for hyperparameter selection only",
    },
    {
        "split"      : "Test (Official Holdout)",
        "start_date" : str(hold_df["date"].min().date()),
        "end_date"   : str(hold_df["date"].max().date()),
        "n_weeks"    : hold_df["date"].nunique(),
        "n_rows"     : len(hold_df),
        "note"       : "Test period: official OOS evaluation; never used in tuning or model selection",
    },
])

print("train_holdout_date_table:")
print(train_holdout_date_table.to_string(index=False))
save_table(train_holdout_date_table, "train_holdout_date_table")

# ── target_balance_table ─────────────────────────────────────────────────────
# Generated here (after split) so we can report per-split positive-class rates.
# A similar positive-class rate across splits confirms no mechanical imbalance.

def _split_stats(df: pd.DataFrame, name: str) -> dict:
    return {
        "split"         : name,
        "positive_rate" : round(df["target_up"].mean(), 4),
        "n_rows"        : len(df),
        "n_weeks"       : df["date"].nunique(),
        "start_date"    : str(df["date"].min().date()),
        "end_date"      : str(df["date"].max().date()),
    }

target_balance_table = pd.DataFrame([
    _split_stats(train_df, "Train"),
    _split_stats(val_df,   "Validation"),
    _split_stats(hold_df,  "Test (Official Holdout)"),
    _split_stats(panel,    "Full Sample"),
])

print("target_balance_table:")
print(target_balance_table.to_string(index=False))
save_table(target_balance_table, "target_balance_table")

print("\nNote: similar positive-class rates across splits confirm no")
print("mechanical imbalance or temporal shift in the classification target.")

### 11.1) Pre-test Hyperparameter Tuning

Hyperparameters are selected once on the 26-week time-ordered validation block using validation net Sharpe at 5 bps. The test period (official holdout) is never examined during this step.

The trailing volatility estimate is seeded from the last 26 training dates before each validation window opens. Logistic Regression is tuned over three values of the regularisation parameter C; the best setting is identified using validation net Sharpe at 5 bps.

In [ ]:
# ── Seed data: last VOL_WINDOW training dates ─────────────────────────────────
# These are used to initialise the trailing vol estimate at the start of validation.
# Using in-sample predictions for seeding is acceptable because the seeded vol
# does NOT affect which hyperparameters we select (that is determined by the
# out-of-sample validation returns only).

_seed_dates = sorted(train_dates_list)[-VOL_WINDOW:]
seed_df = train_df[train_df["date"].isin(set(_seed_dates))].copy()
print(f"Seed dates for vol initialisation: {len(_seed_dates)}  "
      f"({min(_seed_dates).date()} → {max(_seed_dates).date()})")
print(f"seed_df rows: {len(seed_df)}")

# ── Logistic Regression: grid over C ─────────────────────────────────────────
print("Tuning Logistic Regression over C =", LR_C_GRID)
print(f"Validation window: {N_VALIDATION_WEEKS} weeks  |  "
      f"Metric: net Sharpe @{BASE_COST_BPS} bps\n")

lr_results = []
_t0_lr = time.time()

for C in LR_C_GRID:
    model_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
            C=C, max_iter=1000, solver="lbfgs",
            random_state=RANDOM_SEED)),
    ])
    sharpe = _val_net_sharpe(model_lr, train_df, seed_df, val_df)
    lr_results.append({"C": C, "val_net_sharpe": round(sharpe, 4)})
    print(f"  C = {C:5.1f}  →  val net Sharpe = {sharpe:+.4f}")

_lr_runtime = time.time() - _t0_lr

# Select best
lr_results_df = pd.DataFrame(lr_results)
_best_lr_idx  = lr_results_df["val_net_sharpe"].idxmax()
BEST_LR_C     = float(lr_results_df.loc[_best_lr_idx, "C"])
BEST_LR_SHARPE = float(lr_results_df.loc[_best_lr_idx, "val_net_sharpe"])

print(f"\n→ Selected C = {BEST_LR_C}  (val net Sharpe = {BEST_LR_SHARPE:+.4f})")
print(f"  LR tuning runtime: {_lr_runtime:.2f}s")

In [ ]:
# ── Random Forest: grid over max_depth × min_samples_leaf ────────────────────
print("Tuning Random Forest over max_depth =", RF_MAX_DEPTH_GRID,
      "× min_samples_leaf =", RF_MIN_LEAF_GRID)
print(f"n_estimators = {RF_N_ESTIMATORS} (fixed)  |  "
      f"max_features = '{RF_MAX_FEATURES}' (fixed)")
print(f"Validation window: {N_VALIDATION_WEEKS} weeks  |  "
      f"Metric: net Sharpe @{BASE_COST_BPS} bps\n")

rf_results = []
_t0_rf = time.time()

for depth, leaf in product(RF_MAX_DEPTH_GRID, RF_MIN_LEAF_GRID):
    model_rf = RandomForestClassifier(
        n_estimators     = RF_N_ESTIMATORS,
        max_depth        = depth,
        min_samples_leaf = leaf,
        max_features     = RF_MAX_FEATURES,
        random_state     = RANDOM_SEED,
        n_jobs           = -1,
    )
    sharpe = _val_net_sharpe(model_rf, train_df, seed_df, val_df)
    rf_results.append({
        "max_depth"       : depth,
        "min_samples_leaf": leaf,
        "val_net_sharpe"  : round(sharpe, 4),
    })
    print(f"  depth={depth}, min_leaf={leaf:2d}  →  val net Sharpe = {sharpe:+.4f}")

_rf_runtime = time.time() - _t0_rf

rf_results_df = pd.DataFrame(rf_results)
_best_rf_idx  = rf_results_df["val_net_sharpe"].idxmax()
BEST_RF_DEPTH = int(rf_results_df.loc[_best_rf_idx, "max_depth"])
BEST_RF_LEAF  = int(rf_results_df.loc[_best_rf_idx, "min_samples_leaf"])
BEST_RF_SHARPE = float(rf_results_df.loc[_best_rf_idx, "val_net_sharpe"])

print(f"\n→ Selected depth={BEST_RF_DEPTH}, min_leaf={BEST_RF_LEAF}  "
      f"(val net Sharpe = {BEST_RF_SHARPE:+.4f})")
print(f"  RF tuning runtime: {_rf_runtime:.1f}s")

The selected hyperparameters are frozen before any holdout data are examined. `model_spec_table` records the chosen settings for both models.

In [ ]:
# ── These constants are now FROZEN ────────────────────────────────────────────
# BEST_LR_C, BEST_RF_DEPTH, BEST_RF_LEAF must not be changed after this point.
# The official holdout was never examined during tuning.
print("="*55)
print("FROZEN HYPERPARAMETERS (not changed after this point)")
print(f"  Logistic Regression : C = {BEST_LR_C}")
print(f"  Random Forest       : max_depth = {BEST_RF_DEPTH}, "
      f"min_samples_leaf = {BEST_RF_LEAF}")
print("="*55)

# ── pre_holdout_tuning_summary_table ─────────────────────────────────────────
pre_holdout_tuning_summary_table = pd.DataFrame([
    {
        "model"            : "Logistic Regression",
        "candidate_params" : f"C = {LR_C_GRID}",
        "n_candidates"     : len(LR_C_GRID),
        "val_weeks"        : N_VALIDATION_WEEKS,
        "selection_metric" : f"net Sharpe @{BASE_COST_BPS} bps",
        "chosen_params"    : f"C = {BEST_LR_C}",
        "val_net_sharpe"   : round(BEST_LR_SHARPE, 4),
        "runtime_sec"      : round(_lr_runtime, 2),
    },
    {
        "model"            : "Random Forest",
        "candidate_params" : (f"max_depth={RF_MAX_DEPTH_GRID}, "
                              f"min_leaf={RF_MIN_LEAF_GRID}"),
        "n_candidates"     : len(RF_MAX_DEPTH_GRID) * len(RF_MIN_LEAF_GRID),
        "val_weeks"        : N_VALIDATION_WEEKS,
        "selection_metric" : f"net Sharpe @{BASE_COST_BPS} bps",
        "chosen_params"    : f"depth={BEST_RF_DEPTH}, min_leaf={BEST_RF_LEAF}",
        "val_net_sharpe"   : round(BEST_RF_SHARPE, 4),
        "runtime_sec"      : round(_rf_runtime, 2),
    },
])

print("\npre_holdout_tuning_summary_table:")
print(pre_holdout_tuning_summary_table.to_string(index=False))
save_table(pre_holdout_tuning_summary_table, "pre_holdout_tuning_summary_table")

# ── model_spec_table ──────────────────────────────────────────────────────────
model_spec_table = pd.DataFrame([
    {
        "model"       : "Logistic Regression",
        "base_spec"   : "Pipeline(StandardScaler → LogisticRegression(solver=lbfgs))",
        "tuning_rule" : ("C chosen on 26-week time-ordered val block; "
                         "frozen during official holdout"),
        "chosen_params": f"C = {BEST_LR_C}",
        "n_estimators" : "N/A",
        "random_state" : RANDOM_SEED,
        "leakage_note" : "StandardScaler fitted inside Pipeline on train rows only",
    },
    {
        "model"       : "Random Forest",
        "base_spec"   : (f"RandomForestClassifier(n_estimators={RF_N_ESTIMATORS}, "
                         f"max_features='{RF_MAX_FEATURES}')"),
        "tuning_rule" : ("max_depth + min_samples_leaf chosen on 26-week val block; "
                         "frozen during official holdout"),
        "chosen_params": f"max_depth={BEST_RF_DEPTH}, min_samples_leaf={BEST_RF_LEAF}",
        "n_estimators" : RF_N_ESTIMATORS,
        "random_state" : RANDOM_SEED,
        "leakage_note" : "Tree model does not require feature scaling; no scaler or imputer is fitted on the full sample.",
    },
])

print("model_spec_table:")
print(model_spec_table[["model","chosen_params","tuning_rule"]].to_string(index=False))
save_table(model_spec_table, "model_spec_table")

### 11.2) Walk-Forward Prediction Engine

Monthly expanding-window refit: the model refits on the first test-period Friday of each calendar month. At each weekly decision date, predicted probabilities are generated for all `FINAL_TICKERS`.

Monthly refit dates are defined as the first holdout Friday of each calendar month. The walk-forward loop then iterates over all 104 holdout dates, refitting on schedule and generating predicted probabilities for all `FINAL_TICKERS` at each step.

In [ ]:
# ── Monthly refit schedule ────────────────────────────────────────────────────
# First holdout Friday of each calendar month = refit date.
from collections import defaultdict

_month_firsts: dict = defaultdict(list)
for dt in holdout_dates:
    _month_firsts[(dt.year, dt.month)].append(dt)

monthly_refit_dates: set = {min(v) for v in _month_firsts.values()}

print(f"Monthly refit dates : {len(monthly_refit_dates)}")
print(f"Holdout dates total : {len(holdout_dates)}")
print(f"Dates between refits: {len(holdout_dates) - len(monthly_refit_dates)}")
print("\nFirst 6 refit dates:", sorted(monthly_refit_dates)[:6])

# ── Official walk-forward prediction loop ─────────────────────────────────────
# The audit table records the ACTUAL training window used by the currently
# fitted model, not the hypothetical max-available window at each decision date.
# For non-refit weeks, model_train_* reflects the last refit date's training set.

score_records_lr: dict = {}
score_records_rf: dict = {}
audit_records:    list = []

_current_lr = None
_current_rf = None

# State tracking for audit (actual model training window)
_current_train_min_date = None
_current_train_max_date = None
_current_train_rows     = None

_t0_wf = time.time()

for _i, decision_date in enumerate(holdout_dates):

    # ── All rows available for training at this date ──────────────────────────
    available_rows = panel[panel["date"] < decision_date]
    available_train_max = available_rows["date"].max()

    # ── Monthly refit ─────────────────────────────────────────────────────────
    is_refit = (decision_date in monthly_refit_dates) or (_current_lr is None)

    if is_refit:
        X_tr = available_rows[FEATURE_COLS]
        y_tr = available_rows["target_up"]

        _current_lr = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(
                C            = BEST_LR_C,
                max_iter     = 1000,
                solver       = "lbfgs",
                random_state = RANDOM_SEED,
            )),
        ])
        _current_lr.fit(X_tr, y_tr)

        _current_rf = RandomForestClassifier(
            n_estimators     = RF_N_ESTIMATORS,
            max_depth        = BEST_RF_DEPTH,
            min_samples_leaf = BEST_RF_LEAF,
            max_features     = RF_MAX_FEATURES,
            random_state     = RANDOM_SEED,
            n_jobs           = -1,
        )
        _current_rf.fit(X_tr, y_tr)

        # Record the ACTUAL training window used at this refit
        _current_train_min_date = available_rows["date"].min()
        _current_train_max_date = available_rows["date"].max()
        _current_train_rows     = len(available_rows)

    # ── Test data: exactly this decision_date ─────────────────────────────────
    test_rows = panel[panel["date"] == decision_date]
    X_te      = test_rows[FEATURE_COLS]

    # ── Predict probabilities ─────────────────────────────────────────────────
    proba_lr = _current_lr.predict_proba(X_te)[:, 1]
    proba_rf = _current_rf.predict_proba(X_te)[:, 1]

    tickers_te = test_rows["ticker"].values
    score_records_lr[decision_date] = pd.Series(proba_lr, index=tickers_te)
    score_records_rf[decision_date] = pd.Series(proba_rf, index=tickers_te)

    # ── Audit record: actual model training window ────────────────────────────
    audit_records.append({
        "decision_date"           : decision_date,
        "available_train_max_date": available_train_max,
        "model_train_min_date"    : _current_train_min_date,
        "model_train_max_date"    : _current_train_max_date,
        "model_train_rows"        : _current_train_rows,
        "test_rows"               : len(test_rows),
        "n_assets"                : len(test_rows),
        "model_train_max_lt_decision": bool(_current_train_max_date < decision_date),
        "refit_flag"              : is_refit,
        "models"                  : "Logistic Regression + Random Forest",
    })

    # Progress every 20 weeks
    if (_i + 1) % 20 == 0 or (_i + 1) == len(holdout_dates):
        elapsed = time.time() - _t0_wf
        print(f"  [{_i+1:3d}/{len(holdout_dates)}]  {decision_date.date()}  "
              f"model_train_rows={_current_train_rows:,}  "
              f"n_assets={len(test_rows)}  "
              f"refit={is_refit}  elapsed={elapsed:.1f}s")

print(f"\nWalk-forward complete.  Total time: {time.time()-_t0_wf:.1f}s")

Score panels are assembled into DataFrames indexed by date. The runtime audit table records the actual training window used by the fitted model at each holdout date.

In [ ]:
# ── Convert score records to DataFrames ──────────────────────────────────────
score_panel_lr = (pd.DataFrame(score_records_lr)
                  .T
                  .reindex(columns=FINAL_TICKERS))
score_panel_lr.index.name = "date"

score_panel_rf = (pd.DataFrame(score_records_rf)
                  .T
                  .reindex(columns=FINAL_TICKERS))
score_panel_rf.index.name = "date"

print(f"score_panel_lr  shape : {score_panel_lr.shape}")
print(f"score_panel_rf  shape : {score_panel_rf.shape}")
print(f"LR  score range : [{score_panel_lr.values.min():.4f}, "
      f"{score_panel_lr.values.max():.4f}]")
print(f"RF  score range : [{score_panel_rf.values.min():.4f}, "
      f"{score_panel_rf.values.max():.4f}]")

# ── Assertions ────────────────────────────────────────────────────────────────
assert score_panel_lr.shape == (N_HOLDOUT_WEEKS, len(FINAL_TICKERS)), (
    f"score_panel_lr shape mismatch: {score_panel_lr.shape} vs "
    f"({N_HOLDOUT_WEEKS}, {len(FINAL_TICKERS)})")
assert score_panel_rf.shape == (N_HOLDOUT_WEEKS, len(FINAL_TICKERS)), (
    f"score_panel_rf shape mismatch: {score_panel_rf.shape} vs "
    f"({N_HOLDOUT_WEEKS}, {len(FINAL_TICKERS)})")

_lr_nan = score_panel_lr.isna().sum().sum()
_rf_nan = score_panel_rf.isna().sum().sum()
assert _lr_nan == 0, f"score_panel_lr contains {_lr_nan} NaN values"
assert _rf_nan == 0, f"score_panel_rf contains {_rf_nan} NaN values"
print("Assertions PASSED: shape and no-NaN checks on both score panels.")

# ── Save score panels ─────────────────────────────────────────────────────────
save_table(score_panel_lr.reset_index(), "score_panel_lr")
save_table(score_panel_rf.reset_index(), "score_panel_rf")

# ── Build official_runtime_audit_table ───────────────────────────────────────
official_runtime_audit_table = pd.DataFrame(audit_records)

for _col in ["decision_date", "available_train_max_date",
             "model_train_min_date", "model_train_max_date"]:
    official_runtime_audit_table[_col] = pd.to_datetime(
        official_runtime_audit_table[_col])

print(f"official_runtime_audit_table: {official_runtime_audit_table.shape}")
print("\nFirst 5 rows:")
print(official_runtime_audit_table.head(5).to_string(index=False))
print("\nLast 5 rows:")
print(official_runtime_audit_table.tail(5).to_string(index=False))
save_table(official_runtime_audit_table, "official_runtime_audit_table")

# ── Walk-forward audit assertions ────────────────────────────────────────────
aud = official_runtime_audit_table

# A1: model_train_max_date < decision_date for ALL rows
# (checks the ACTUAL fitted model, not hypothetical available data)
_all_lt = aud["model_train_max_lt_decision"].all()
assert _all_lt, (
    "LEAKAGE DETECTED: some model_train_max_date >= decision_date. "
    f"Failing rows: "
    f"{aud[~aud['model_train_max_lt_decision']]['decision_date'].tolist()}"
)
print("Assertion A1 PASSED: model_train_max_date < decision_date "
      "for all holdout dates.")

# A2: all holdout dates have n_assets == len(FINAL_TICKERS)
_bad_n = aud[aud["n_assets"] != len(FINAL_TICKERS)]
if len(_bad_n) > 0:
    print(f"WARNING: {len(_bad_n)} dates have n_assets ≠ {len(FINAL_TICKERS)}:")
    print(_bad_n[["decision_date", "n_assets"]].to_string(index=False))
else:
    print(f"Assertion A2 PASSED: all holdout dates have "
          f"n_assets = {len(FINAL_TICKERS)}.")

# A3: official holdout never used in hyperparameter tuning
assert all(
    d not in _validation_set and d not in _train_set
    for d in holdout_dates
), "Holdout date appeared in tuning data!"
print("Assertion A3 PASSED: official holdout never used in tuning.")

# A4: refit count summary
n_refits = int(aud["refit_flag"].sum())
print(f"\nRefit dates          : {n_refits}  "
      f"(out of {len(holdout_dates)} holdout weeks)")
print(f"Avg weeks per refit  : {len(holdout_dates)/n_refits:.1f}  (target: ~4.3)")

# A5: confirm model_train_max is constant between refits
# (non-refit rows should share the same model_train_max as the previous refit)
_refit_rows = aud[aud["refit_flag"]]
print(f"\nModel training window at first refit:")
print(f"  min date : {_refit_rows.iloc[0]['model_train_min_date'].date()}")
print(f"  max date : {_refit_rows.iloc[0]['model_train_max_date'].date()}")
print(f"  rows     : {_refit_rows.iloc[0]['model_train_rows']:,}")
print(f"Model training window at last refit:")
print(f"  min date : {_refit_rows.iloc[-1]['model_train_min_date'].date()}")
print(f"  max date : {_refit_rows.iloc[-1]['model_train_max_date'].date()}")
print(f"  rows     : {_refit_rows.iloc[-1]['model_train_rows']:,}")

## 12) Run Backtests

Raw top/bottom 20% weight panels are built from the score panels, exposure checks confirm the portfolio construction rules, and volatility targeting is applied to produce the **official scaled weights**. The official backtest then converts scaled weights into gross and net returns.

Raw top/bottom 20% weight panels are constructed from the score panels. Exposure checks confirm that each date has long exposure ≈ +1, short exposure ≈ −1, and 18 non-zero positions. The portfolio rule table is saved.

In [ ]:
# ── Build raw weight matrices from score panels ───────────────────────────────
# These are RAW (unscaled) weights. They are used:
#   (a) as inputs to volatility targeting in Section 12, and
#   (b) to compute raw gross returns for the vol estimator.
# ALL official performance metrics in Sections 13+ use SCALED weights (from Section 12).

raw_weights_lr = make_weight_panel_from_scores(score_panel_lr, q=BASE_QUANTILE)
raw_weights_rf = make_weight_panel_from_scores(score_panel_rf, q=BASE_QUANTILE)

print(f"raw_weights_lr shape : {raw_weights_lr.shape}  "
      f"(index=date, columns=FINAL_TICKERS)")
print(f"raw_weights_rf shape : {raw_weights_rf.shape}")

# Label as diagnostic (not used in official results)
print("\n⚠  raw_weights_lr / raw_weights_rf are RAW DIAGNOSTICS only.")
print("   Official results in Sections 13+ use scaled weights from Section 12.")

# ── Reusable exposure check function ─────────────────────────────────────────
def check_weight_exposures(weights_df: pd.DataFrame,
                            model_name: str,
                            q: float = BASE_QUANTILE) -> None:
    """
    Assert and report exposure properties for a weight panel.

    Checks for each holdout date:
      long  exposure ≈ +1
      short exposure ≈ −1
      net   exposure ≈  0
      gross exposure ≈ +2
      non-zero names == 2 × k  (k = floor(N × q))

    Parameters
    ----------
    weights_df : DataFrame (T × N) of portfolio weights.
    model_name : Label for printed output.
    q          : Quantile used to compute expected k.
    """
    k_exp = int(np.floor(weights_df.shape[1] * q))

    long_exp  = weights_df[weights_df > 0].sum(axis=1)
    short_exp = weights_df[weights_df < 0].sum(axis=1)
    net_exp   = weights_df.sum(axis=1)
    gross_exp = weights_df.abs().sum(axis=1)
    n_nonzero = (weights_df != 0).sum(axis=1)

    print(f"\nExposure checks — {model_name}  "
          f"(shape {weights_df.shape[0]}×{weights_df.shape[1]}):")
    print(f"  Long  : mean={long_exp.mean():.4f}  "
          f"min={long_exp.min():.4f}  max={long_exp.max():.4f}  [target ≈ +1.0]")
    print(f"  Short : mean={short_exp.mean():.4f}  "
          f"min={short_exp.min():.4f}  max={short_exp.max():.4f}  [target ≈ −1.0]")
    print(f"  Net   : mean={net_exp.mean():.6f}  "
          f"max|net|={net_exp.abs().max():.2e}  [target ≈ 0.0]")
    print(f"  Gross : mean={gross_exp.mean():.4f}  "
          f"min={gross_exp.min():.4f}  max={gross_exp.max():.4f}  [target ≈ 2.0]")
    print(f"  Non-zero names: mean={n_nonzero.mean():.1f}  "
          f"mode={n_nonzero.mode()[0]}  [target = {2*k_exp}]")

    # Assertions
    bad_long  = (long_exp  - 1.0).abs() > 1e-9
    bad_short = (short_exp + 1.0).abs() > 1e-9
    bad_net   = net_exp.abs()            > 1e-9
    bad_gross = (gross_exp - 2.0).abs() > 1e-9
    bad_nz    = n_nonzero != 2 * k_exp

    for flag, label in [(bad_long,  "long ≈ +1"),
                        (bad_short, "short ≈ −1"),
                        (bad_net,   "net ≈ 0"),
                        (bad_gross, "gross ≈ 2"),
                        (bad_nz,    f"non-zero = {2*k_exp}")]:
        if flag.any():
            print(f"  WARNING ({label}): {flag.sum()} dates fail.")
        else:
            print(f"  ✓ {label} on all dates")


# ── Apply to both raw weight panels ──────────────────────────────────────────
check_weight_exposures(raw_weights_lr, "LR raw weights")
check_weight_exposures(raw_weights_rf, "RF raw weights")


# ── portfolio_rule_table ─────────────────────────────────────────────────────
portfolio_rule_table = pd.DataFrame([
    {"item": "Universe size",
     "definition": f"{len(FINAL_TICKERS)} final tickers (FINAL_TICKERS)"},
    {"item": "Long leg",
     "definition": f"Top {int(BASE_QUANTILE*100)}% by predicted probability "
                   f"({int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE))} names)"},
    {"item": "Short leg",
     "definition": f"Bottom {int(BASE_QUANTILE*100)}% by predicted probability "
                   f"({int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE))} names)"},
    {"item": "Non-selected",
     "definition": "Weight = 0 (27 names per date)"},
    {"item": "Weighting",
     "definition": "Equal-weight within each leg"},
    {"item": "Long weight per name",
     "definition": f"+1/{int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE))} "
                   f"≈ +{1/int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE)):.4f}"},
    {"item": "Short weight per name",
     "definition": f"−1/{int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE))} "
                   f"≈ −{1/int(np.floor(len(FINAL_TICKERS)*BASE_QUANTILE)):.4f}"},
    {"item": "Gross exposure",
     "definition": "+2 (raw); scaled by vol-targeting scaler in Section 12"},
    {"item": "Net exposure",
     "definition": "0 (dollar-neutral)"},
    {"item": "Rebalance frequency",
     "definition": "Weekly at Friday close (same date as signal generation)"},
    {"item": "Execution assumption",
     "definition": "Trade at Friday close t; portfolio held during t+1"},
    {"item": "Score used for ranking",
     "definition": "Predicted probability of target_up = 1 (class 1 proba)"},
    {"item": "Robustness quantiles tested",
     "definition": "q = 10%, 20% (base), 30% — tested in Section 18"},
])

print("portfolio_rule_table:")
print(portfolio_rule_table.to_string(index=False))
save_table(portfolio_rule_table, "portfolio_rule_table")

The asset-returns panel is aligned to holdout dates. Past-only volatility targeting is applied to both models using the trailing 26-week realised volatility of raw gross returns, producing the official scaled weight panels.

In [ ]:
# ── Asset returns panel aligned to holdout dates ─────────────────────────────
# We need ret_fwd1 per ticker per holdout decision date.
# This is the realized return during the holding period t+1.

_hold_panel = panel[panel["date"].isin(_holdout_set)].copy()
asset_returns_holdout = (
    _hold_panel
    .pivot(index="date", columns="ticker", values="ret_fwd1")
    .reindex(index=holdout_dates, columns=FINAL_TICKERS)
)

print(f"asset_returns_holdout shape : {asset_returns_holdout.shape}")
print(f"NaN count : {asset_returns_holdout.isna().sum().sum()}")
if asset_returns_holdout.isna().sum().sum() > 0:
    print("WARNING: NaN ret_fwd1 values present — these will be filled with 0.0 "
          "inside apply_vol_targeting().")

# ── Apply vol targeting: Logistic Regression ─────────────────────────────────
(scaled_weights_lr,
 scaler_lr,
 raw_gross_returns_lr) = apply_vol_targeting(
     raw_weights_df   = raw_weights_lr,
     asset_returns_df = asset_returns_holdout,
)

print(f"LR  scaler: mean={scaler_lr.mean():.3f}  min={scaler_lr.min():.3f}  max={scaler_lr.max():.3f}  pct<1={( scaler_lr < 1).mean():.0%}")

# ── Apply vol targeting: Random Forest ───────────────────────────────────────
(scaled_weights_rf,
 scaler_rf,
 raw_gross_returns_rf) = apply_vol_targeting(
     raw_weights_df   = raw_weights_rf,
     asset_returns_df = asset_returns_holdout,
)

print(f"RF  scaler: mean={scaler_rf.mean():.3f}  min={scaler_rf.min():.3f}  max={scaler_rf.max():.3f}  pct<1={(scaler_rf < 1).mean():.0%}")
# ── Save outputs: scaled weights and scalers ─────────────────────────────────
# Scaled weights are the OFFICIAL strategy weights.
print("\nSaving official outputs...")
save_table(scaled_weights_lr.reset_index(), "scaled_weights_lr")
save_table(scaled_weights_rf.reset_index(), "scaled_weights_rf")
save_table(scaler_lr.reset_index(name="scaler").rename(
    columns={scaler_lr.index.name or "index": "date"}), "scaler_lr")
save_table(scaler_rf.reset_index(name="scaler").rename(
    columns={scaler_rf.index.name or "index": "date"}), "scaler_rf")

# Raw weights saved as clearly labelled diagnostics
save_table(raw_weights_lr.reset_index(), "raw_weights_lr_diagnostic")
save_table(raw_weights_rf.reset_index(), "raw_weights_rf_diagnostic")
print("All weight/scaler outputs saved.")


# ── Verify scaled weights have no NaN ────────────────────────────────────────
assert scaled_weights_lr.isna().sum().sum() == 0, "NaN in scaled_weights_lr"
assert scaled_weights_rf.isna().sum().sum() == 0, "NaN in scaled_weights_rf"
assert scaler_lr.isna().sum() == 0, "NaN in scaler_lr"
assert scaler_rf.isna().sum() == 0, "NaN in scaler_rf"
print("Checks PASSED: no NaN in scaled weights or scalers.")

# ── Confirm scaler is bounded ─────────────────────────────────────────────────
assert scaler_lr.between(*SCALER_CLIP).all(), "scaler_lr out of clip bounds"
assert scaler_rf.between(*SCALER_CLIP).all(), "scaler_rf out of clip bounds"
print(f"Checks PASSED: all scalers within {SCALER_CLIP}.")

# ── Note: scaled_weights are the OFFICIAL strategy weights ───────────────────
print("\nOfficial strategy from Section 13 onward uses:")
print("  scaled_weights_lr  ← Logistic Regression official weights")
print("  scaled_weights_rf  ← Random Forest official weights")
print("Raw weights (raw_weights_lr, raw_weights_rf) are DIAGNOSTIC ONLY.")

The official OOS backtest converts scaled weights into gross and net return series using drifted one-way turnover. Post-backtest assertions confirm observation counts, no missing values, and that scaled weights were used throughout.

In [ ]:
# ── Official backtest: Logistic Regression ───────────────────────────────────
# Uses scaled_weights_lr (NOT raw_weights_lr)
print("Running official backtest — Logistic Regression...")
_bt_lr = run_official_backtest_from_weights(
    scaled_weights_df = scaled_weights_lr,     # ← SCALED weights
    asset_returns_df  = asset_returns_holdout,
    cost_bps          = BASE_COST_BPS,
    label             = "LR",
)

gross_returns_lr = _bt_lr["gross_returns"]
net_returns_lr   = _bt_lr["net_returns"]
turnover_lr      = _bt_lr["turnover"]

# ── Official backtest: Random Forest ─────────────────────────────────────────
print("\nRunning official backtest — Random Forest...")
_bt_rf = run_official_backtest_from_weights(
    scaled_weights_df = scaled_weights_rf,     # ← SCALED weights
    asset_returns_df  = asset_returns_holdout,
    cost_bps          = BASE_COST_BPS,
    label             = "RF",
)

gross_returns_rf = _bt_rf["gross_returns"]
net_returns_rf   = _bt_rf["net_returns"]
turnover_rf      = _bt_rf["turnover"]

# ── Collect for convenience ───────────────────────────────────────────────────
backtest_results = {
    "lr": _bt_lr,
    "rf": _bt_rf,
}

print("\nOfficial backtest complete.")

# ── Check 1: correct number of observations ──────────────────────────────────
for _name, _s in [("gross_returns_lr", gross_returns_lr),
                  ("net_returns_lr",   net_returns_lr),
                  ("turnover_lr",      turnover_lr),
                  ("gross_returns_rf", gross_returns_rf),
                  ("net_returns_rf",   net_returns_rf),
                  ("turnover_rf",      turnover_rf)]:
    assert len(_s) == N_HOLDOUT_WEEKS, (
        f"{_name}: expected {N_HOLDOUT_WEEKS} observations, got {len(_s)}")
print(f"Check 1 PASSED: all series have {N_HOLDOUT_WEEKS} observations.")

# ── Check 2: no NaN values ────────────────────────────────────────────────────
for _name, _s in [("gross_returns_lr", gross_returns_lr),
                  ("net_returns_lr",   net_returns_lr),
                  ("turnover_lr",      turnover_lr),
                  ("gross_returns_rf", gross_returns_rf),
                  ("net_returns_rf",   net_returns_rf),
                  ("turnover_rf",      turnover_rf)]:
    assert _s.isna().sum() == 0, f"{_name}: NaN values detected"
print("Check 2 PASSED: no NaN in any return or turnover series.")

# ── Check 3: turnover is finite and non-negative ──────────────────────────────
assert (turnover_lr >= 0).all() and np.isfinite(turnover_lr).all(),     "turnover_lr has negative or infinite values"
assert (turnover_rf >= 0).all() and np.isfinite(turnover_rf).all(),     "turnover_rf has negative or infinite values"
print(f"Check 3 PASSED: turnover is finite and non-negative.")
print(f"  LR avg turnover : {turnover_lr.mean():.4f}")
print(f"  RF avg turnover : {turnover_rf.mean():.4f}")

# ── Check 4: official backtest used SCALED weights ────────────────────────────
# Verify by checking that scaled_weights_lr was passed — we confirm indirectly:
# gross_returns computed inside run_official_backtest_from_weights differ from
# raw gross returns when any scaler ≠ 1.
_raw_gross_check = (raw_weights_lr * asset_returns_holdout.fillna(0)).sum(axis=1)
_scaled_gross_check = (scaled_weights_lr * asset_returns_holdout.fillna(0)).sum(axis=1)
_n_differ = ((_raw_gross_check - _scaled_gross_check).abs() > 1e-12).sum()
print(f"\nCheck 4: official gross returns differ from raw gross returns on "
      f"{_n_differ}/{N_HOLDOUT_WEEKS} dates  "
      f"(expected > 0, confirming scaled weights were used).")
if _n_differ == 0:
    print("  WARNING: scaled and raw gross returns are identical — "
          "scaler may always = 1. Check vol targeting output.")

# ── Check 5: full FINAL_TICKERS vector used in backtest ─────────────────────
# run_official_backtest_from_weights() reindexes to FINAL_TICKERS internally.
# Verify here that the scaled weight panels match the required shape.
_lr_cols = set(scaled_weights_lr.columns.tolist())
_rf_cols = set(scaled_weights_rf.columns.tolist())
_ft_set  = set(FINAL_TICKERS)
assert _lr_cols == _ft_set, (
    f"scaled_weights_lr columns do not match FINAL_TICKERS. "
    f"Extra: {_lr_cols - _ft_set}  Missing: {_ft_set - _lr_cols}")
assert _rf_cols == _ft_set, (
    f"scaled_weights_rf columns do not match FINAL_TICKERS. "
    f"Extra: {_rf_cols - _ft_set}  Missing: {_ft_set - _rf_cols}")
assert scaled_weights_lr.shape[1] == len(FINAL_TICKERS)
assert scaled_weights_rf.shape[1] == len(FINAL_TICKERS)
print(f"Check 5 PASSED: scaled weight panels have {len(FINAL_TICKERS)} columns "
      f"matching FINAL_TICKERS exactly — full vector confirmed.")

## 13) Turnover Definition

The official backtest converts **scaled weights** into weekly gross and net returns.

| Step | Computation |
|------|-------------|
| Gross return at t | `Σ_i w_{i,t} × ret_fwd1_{i,t}` (scaled weights × realized returns) |
| Drifted weight | `w⁺_{i,t-1} = w_{i,t-1} × (1 + r_{i,t-1}) / (1 + r_{port,t-1})` |
| One-way turnover | `0.5 × Σ_i |w_{i,t} − w⁺_{i,t-1}|` (full 45-ticker vector) |
| Net return at t | `gross_return_t − turnover_t × cost_bps / 10 000` |
| First week | Turnover = `0.5 × Σ_i |w_{i,0}|` (no prior holdings) |

All computations use the **full FINAL_TICKERS vector** including zero-weight names.
Excluding zero-weight names from turnover would understate switching costs
when a name enters or exits the portfolio.

All computations use the **full FINAL_TICKERS vector** including zero-weight names. Excluding zero-weight names from turnover would understate switching costs when a name enters or exits the portfolio. The functions `compute_drifted_weights` and `compute_one_way_turnover` (defined in section 10) implement this formula exactly.

## 14) Results Summary

The official tear sheet and cost sensitivity table report performance under the base-case 5 bps assumption and at 1 and 10 bps. All results are based on volatility-targeted scaled weights.

**Official tear sheet** — gross and net performance at 5 bps:

In [ ]:
# ── official_tearsheet_table (scaled strategy, 5 bps) ────────────────────────
_ts_lr = make_tearsheet(gross_returns_lr, net_returns_lr, turnover_lr,
                        "Logistic Regression")
_ts_rf = make_tearsheet(gross_returns_rf, net_returns_rf, turnover_rf,
                        "Random Forest")

official_tearsheet_table = pd.DataFrame([_ts_lr, _ts_rf])

save_table(official_tearsheet_table, "official_tearsheet_table")
show_table(official_tearsheet_table)

**Cost sensitivity** — net performance at 1, 5, and 10 bps:

In [ ]:
# ── Build cost_sensitivity_table ─────────────────────────────────────────────
_sens_rows = []

for _model_label, _gross_r, _net_base_r, _to_s in [
    ("Logistic Regression", gross_returns_lr, net_returns_lr, turnover_lr),
    ("Random Forest",       gross_returns_rf, net_returns_rf, turnover_rf),
]:
    for _bps in SENSITIVITY_COSTS_BPS:
        # Recompute net returns at this cost level
        _net_r = _gross_r - _to_s * _bps / 10_000

        _sens_rows.append({
            "model"          : _model_label,
            "cost_bps"       : _bps,
            "net_ann_ret"    : round(annualized_return(_net_r),     4),
            "net_sharpe"     : round(sharpe_ratio(_net_r),          4),
            "net_max_dd"     : round(max_drawdown(_net_r),          4),
            "avg_turnover"   : round(float(_to_s.mean()),           4),
            "note"           : ("base case" if _bps == BASE_COST_BPS
                                else "sensitivity"),
        })

cost_sensitivity_table = pd.DataFrame(_sens_rows)

save_table(cost_sensitivity_table, "cost_sensitivity_table")
show_table(cost_sensitivity_table[["model","cost_bps","net_ann_ret","net_sharpe","net_max_dd","avg_turnover"]])

### Cost Sensitivity

Higher turnover amplifies cost drag. The model with greater average turnover shows steeper net-return degradation across cost levels. At 10 bps, both strategies are materially negative, setting the upper bound for a realistic cost stress scenario.

## 15) Benchmark Comparison

SPY (buy-and-hold) and an equal-weight long-only universe serve as passive baselines. Both are computed over the same 104 holdout holding-period dates with no transaction cost assumption.

SPY and equal-weight universe returns are aligned to the same 104 holdout holding-period dates as the strategy. Neither benchmark carries a transaction cost assumption.

In [ ]:
# ── SPY buy-and-hold benchmark ───────────────────────────────────────────────
# spy_ret_weekly was computed in Section 2. Align to holdout decision dates.
# Each holdout date t holds SPY over t → t+1 (ret_fwd1 for SPY).
# Returns must NOT be forward-filled: missing returns would create artificial
# performance and violate the integrity of the benchmark.

spy_ret_fwd1 = spy_ret_weekly.shift(-1)   # SPY return during next week
spy_returns  = spy_ret_fwd1.reindex(holdout_dates)

if spy_returns.isna().any():
    raise ValueError(
        "SPY benchmark has missing t+1 returns. "
        "Adjust DATA_END or holdout_dates; do not forward-fill returns."
    )

print(f"SPY returns aligned: {len(spy_returns)} observations, 0 NaN.")
print(f"SPY return range: [{spy_returns.min():.4f}, {spy_returns.max():.4f}]")

# ── Equal-weight universe long-only benchmark ────────────────────────────────
# At each holdout date t, hold all FINAL_TICKERS equally.
# EW return at t = mean(ret_fwd1 across FINAL_TICKERS) at date t.
# Returns must NOT be forward-filled.

ew_returns = (
    asset_returns_holdout           # (T × N) ret_fwd1 panel
    .mean(axis=1)                   # cross-sectional mean per date
)

if ew_returns.isna().any():
    raise ValueError(
        "Equal-weight benchmark has missing t+1 returns. "
        "Check asset_returns_holdout alignment; do not forward-fill returns."
    )

print(f"EW universe returns aligned: {len(ew_returns)} observations, 0 NaN.")
print(f"EW return range: [{ew_returns.min():.4f}, {ew_returns.max():.4f}]")

# ── Alignment check ──────────────────────────────────────────────────────────
assert list(spy_returns.index) == holdout_dates,     "SPY returns not aligned to holdout_dates"
assert list(ew_returns.index)  == holdout_dates,     "EW returns not aligned to holdout_dates"
assert len(spy_returns) == N_HOLDOUT_WEEKS,     f"SPY: expected {N_HOLDOUT_WEEKS} obs, got {len(spy_returns)}"
assert len(ew_returns)  == N_HOLDOUT_WEEKS,     f"EW: expected {N_HOLDOUT_WEEKS} obs, got {len(ew_returns)}"
print("Alignment checks PASSED: both benchmarks cover exactly the "
      f"{N_HOLDOUT_WEEKS} holdout dates.")

`benchmark_summary_table` and `drawdown_summary_table` are computed using the same production metric functions applied to the strategy results.

In [ ]:
# ── benchmark_summary_table (built here using final production functions) ─────
# Built here — after annualized_return(), annualized_volatility(),
# sharpe_ratio(), and max_drawdown() are defined in Section 10.
# This is the single authoritative version of the benchmark table.

benchmark_summary_table = pd.DataFrame([
    {
        "benchmark"   : "SPY Buy-and-Hold",
        "ann_return"  : round(annualized_return(spy_returns),    4),
        "ann_vol"     : round(annualized_volatility(spy_returns),4),
        "sharpe"      : round(sharpe_ratio(spy_returns),         4),
        "max_drawdown": round(max_drawdown(spy_returns),         4),
        "note"        : "Market-level passive; no transaction cost",
    },
    {
        "benchmark"   : "Equal-Weight Universe",
        "ann_return"  : round(annualized_return(ew_returns),    4),
        "ann_vol"     : round(annualized_volatility(ew_returns),4),
        "sharpe"      : round(sharpe_ratio(ew_returns),         4),
        "max_drawdown": round(max_drawdown(ew_returns),         4),
        "note"        : "Long-only equal-weight across FINAL_TICKERS; no cost",
    },
])

save_table(benchmark_summary_table, "benchmark_summary_table")
show_table(benchmark_summary_table)

# ── drawdown_summary_table ───────────────────────────────────────────────────
def _drawdown_row(name, r):
    dd = drawdown_series(r)
    return {
        "series"          : name,
        "max_drawdown"    : round(float(dd.min()), 4),
        "date_of_max_dd"  : str(dd.idxmin().date()),
        "avg_drawdown"    : round(float(dd.mean()), 4),
        "pct_weeks_in_dd" : round(float((dd < 0).mean()), 4),
    }

drawdown_summary_table = pd.DataFrame([
    _drawdown_row("LR net 5bps (scaled)",  net_returns_lr),
    _drawdown_row("RF net 5bps (scaled)",  net_returns_rf),
    _drawdown_row("SPY buy-and-hold",      spy_returns),
    _drawdown_row("EW universe long-only", ew_returns),
])

save_table(drawdown_summary_table, "drawdown_summary_table")
show_table(drawdown_summary_table[["series","max_drawdown","avg_drawdown","pct_weeks_in_dd"]])

## 16) Performance Plots

The figures below use the official scaled strategy returns throughout. All figures are also saved to `./outputs/figures/`.

**Figure 1 — Cumulative Wealth**

In [ ]:
# ── Figure 1: Official OOS Cumulative Returns ────────────────────────────────
# Official scaled strategy vs SPY and equal-weight universe.

fig1, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(holdout_dates, cumulative_wealth(net_returns_lr).values,
         color=PALETTE["lr"],  label="Logistic Regression (net, 5 bps)", lw=2.0)
ax1.plot(holdout_dates, cumulative_wealth(net_returns_rf).values,
         color=PALETTE["rf"],  label="Random Forest (net, 5 bps)",        lw=2.0)
ax1.plot(holdout_dates, cumulative_wealth(spy_returns).values,
         color=PALETTE["spy"], label="SPY buy-and-hold",                   lw=1.8, ls="--")
ax1.plot(holdout_dates, cumulative_wealth(ew_returns).values,
         color=PALETTE["ew"],  label="Equal-weight universe",              lw=1.8, ls=":")

ax1.axhline(1.0, color="grey", lw=0.8, ls="-", alpha=0.6)
ax1.set_title(
    f"Cumulative Wealth — Official OOS Period\n"
    f"{holdout_dates[0].strftime('%b %Y')} to {holdout_dates[-1].strftime('%b %Y')}, "
    f"5 bps one-way cost",
    fontsize=12)
ax1.set_xlabel("Date")
ax1.set_ylabel("Cumulative Wealth (start = 1.0)")
ax1.legend(loc="upper left", framealpha=0.9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
fig1.autofmt_xdate()
fig1.tight_layout()
save_figure(fig1, "fig_oos_cumulative_returns")
_ipy_display(_IPImage(filename=str(FIG_DIR / "fig_oos_cumulative_returns.png")))

**Figure 2 — Gross vs Net**

In [ ]:
# ── Figure 2: Gross vs Net ───────────────────────────────────────────────────
# Cost drag: difference between gross and net cumulative wealth.

fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, gross_r, net_r, model_label, gc, nc in [
    (axes2[0], gross_returns_lr, net_returns_lr,
     "Logistic Regression", PALETTE["lr_gross"], PALETTE["lr"]),
    (axes2[1], gross_returns_rf, net_returns_rf,
     "Random Forest",       PALETTE["rf_gross"], PALETTE["rf"]),
]:
    ax.plot(holdout_dates, cumulative_wealth(gross_r).values,
            color=gc, ls="--", label="Gross", lw=1.8)
    ax.plot(holdout_dates, cumulative_wealth(net_r).values,
            color=nc, label="Net (5 bps)", lw=2.0)
    ax.axhline(1.0, color="grey", lw=0.8, alpha=0.6)
    ax.set_title(f"{model_label}\nGross vs Net Cumulative Return", fontsize=11)
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative Wealth (start = 1.0)")
    ax.legend(framealpha=0.9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))

fig2.autofmt_xdate()
fig2.tight_layout()
save_figure(fig2, "fig_gross_vs_net")
_ipy_display(_IPImage(filename=str(FIG_DIR / "fig_gross_vs_net.png")))

**Figure 3 — Rolling Drawdown**

In [ ]:
# ── Figure 3: Drawdown Comparison ───────────────────────────────────────────
# Clean line plot: LR, RF, SPY, EW — same palette and linestyles as Fig 1.

fig3, ax3 = plt.subplots(figsize=(12, 4))

ax3.plot(holdout_dates, drawdown_series(net_returns_lr).values,
         color=PALETTE["lr"],  lw=2.0, label="Logistic Regression (net, 5 bps)")
ax3.plot(holdout_dates, drawdown_series(net_returns_rf).values,
         color=PALETTE["rf"],  lw=2.0, label="Random Forest (net, 5 bps)")
ax3.plot(holdout_dates, drawdown_series(spy_returns).values,
         color=PALETTE["spy"], lw=1.8, ls="--", label="SPY buy-and-hold")
ax3.plot(holdout_dates, drawdown_series(ew_returns).values,
         color=PALETTE["ew"],  lw=1.8, ls=":",  label="Equal-weight universe")

ax3.axhline(0, color="grey", lw=0.8, ls="-", alpha=0.6)
ax3.set_title("Drawdown Comparison — Official OOS Period", fontsize=12)
ax3.set_xlabel("Date")
ax3.set_ylabel("Drawdown")
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax3.legend(loc="lower left", framealpha=0.9)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
fig3.autofmt_xdate()
fig3.tight_layout()
save_figure(fig3, "fig_rolling_drawdown")
_ipy_display(_IPImage(filename=str(FIG_DIR / "fig_rolling_drawdown.png")))

**Figure 4 — RF Feature Importances**

In [ ]:
# ── Figure 4: Random Forest Feature Importances ─────────────────────────────
# Uses the last fitted _current_rf (most recent refit in the holdout).

try:
    _importances = _current_rf.feature_importances_
    _fi = (pd.Series(_importances, index=FEATURE_COLS)
           .sort_values(ascending=True))   # ascending so largest bar at top

    fig4, ax4 = plt.subplots(figsize=(9, 5))
    ax4.barh(_fi.index, _fi.values, color=PALETTE["rf"], alpha=0.75, edgecolor="white")
    ax4.set_title(
        "Random Forest — Feature Importances\n"
        "(mean decrease in impurity, last refit model)",
        fontsize=12)
    ax4.set_xlabel("Importance (normalised)")
    ax4.set_ylabel("")
    ax4.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"{x:.3f}"))
    fig4.tight_layout()
    save_figure(fig4, "fig_rf_feature_importance")
    _ipy_display(_IPImage(filename=str(FIG_DIR / "fig_rf_feature_importance.png")))

    # Save feature importance table
    rf_feature_importance_table = (
        pd.Series(_importances, index=FEATURE_COLS)
        .sort_values(ascending=False)
        .reset_index()
    )
    rf_feature_importance_table.columns = ["feature", "importance"]
    rf_feature_importance_table["importance"] = (
        rf_feature_importance_table["importance"].round(6))
    save_table(rf_feature_importance_table, "rf_feature_importance_table")
    print("rf_feature_importance_table saved.")
except Exception as e:
    print(f"Feature importance figure skipped: {e}")

**Figure 5 — Cost Sensitivity**

In [ ]:
# ── Figure 5: Net Sharpe by Cost Assumption ──────────────────────────────────

fig5, ax5 = plt.subplots(figsize=(8, 4))

for _model_label, _color, _marker in [
    ("Logistic Regression", PALETTE["lr"], "o"),
    ("Random Forest",       PALETTE["rf"], "s"),
]:
    _rows = cost_sensitivity_table[
        cost_sensitivity_table["model"] == _model_label
    ].sort_values("cost_bps")

    ax5.plot(_rows["cost_bps"], _rows["net_sharpe"],
             color=_color, marker=_marker, ms=8,
             label=_model_label, lw=2.0)
    for _, row in _rows.iterrows():
        ax5.annotate(f"{row['net_sharpe']:.3f}",
                     (row["cost_bps"], row["net_sharpe"]),
                     textcoords="offset points", xytext=(0, 10),
                     ha="center", fontsize=9, color=_color)

ax5.axhline(0, color="grey", lw=0.8, ls="--", alpha=0.7)
ax5.set_title("Net Sharpe Ratio under Alternative Transaction Cost Assumptions", fontsize=12)
ax5.set_xlabel("One-Way Transaction Cost (bps)")
ax5.set_ylabel("Annualised Net Sharpe Ratio")
ax5.set_xticks(SENSITIVITY_COSTS_BPS)
ax5.legend(framealpha=0.9)
fig5.tight_layout()
save_figure(fig5, "fig_cost_sensitivity_net_sharpe")
_ipy_display(_IPImage(filename=str(FIG_DIR / "fig_cost_sensitivity_net_sharpe.png")))

## 17) Risk Control Discussion

Volatility targeting scales raw portfolio weights each week to target 10% annualised volatility. The trailing estimator uses 26 weeks of raw gross returns, shifted by one period to exclude the current week. The scaler is clipped to [0.5, 1.5] to prevent extreme de-leveraging or over-leveraging.

The table below reports scaler statistics over the official holdout.

In [ ]:
# ── risk_control_summary_table (now includes avg_turnover) ──────────────────
risk_control_summary_table = pd.DataFrame([
    {
        "model"           : "Logistic Regression",
        "risk_control"    : "Volatility targeting",
        "target_ann_vol"  : f"{VOL_TARGET_ANN*100:.0f}%",
        "trailing_window" : f"{VOL_WINDOW} weeks",
        "scaler_clip"     : str(SCALER_CLIP),
        "avg_scaler"      : round(float(scaler_lr.mean()),                4),
        "min_scaler"      : round(float(scaler_lr.min()),                 4),
        "max_scaler"      : round(float(scaler_lr.max()),                 4),
        "pct_weeks_lt_1"  : round(float((scaler_lr < 1).mean()),          4),
        "pct_weeks_gt_1"  : round(float((scaler_lr > 1).mean()),          4),
        "pct_weeks_eq_1"  : round(float((scaler_lr == 1.0).mean()),       4),
        "avg_gross_exp"   : round(float(scaled_weights_lr.abs()
                                        .sum(axis=1).mean()),             4),
        "avg_turnover"    : round(float(turnover_lr.mean()),              4),
    },
    {
        "model"           : "Random Forest",
        "risk_control"    : "Volatility targeting",
        "target_ann_vol"  : f"{VOL_TARGET_ANN*100:.0f}%",
        "trailing_window" : f"{VOL_WINDOW} weeks",
        "scaler_clip"     : str(SCALER_CLIP),
        "avg_scaler"      : round(float(scaler_rf.mean()),                4),
        "min_scaler"      : round(float(scaler_rf.min()),                 4),
        "max_scaler"      : round(float(scaler_rf.max()),                 4),
        "pct_weeks_lt_1"  : round(float((scaler_rf < 1).mean()),          4),
        "pct_weeks_gt_1"  : round(float((scaler_rf > 1).mean()),          4),
        "pct_weeks_eq_1"  : round(float((scaler_rf == 1.0).mean()),       4),
        "avg_gross_exp"   : round(float(scaled_weights_rf.abs()
                                        .sum(axis=1).mean()),             4),
        "avg_turnover"    : round(float(turnover_rf.mean()),              4),
    },
])

save_table(risk_control_summary_table, "risk_control_summary_table")
show_table(risk_control_summary_table[["model","avg_scaler","min_scaler","max_scaler","pct_weeks_lt_1","avg_turnover"]])

## 18) Robustness Checks

Two robustness checks test whether results depend on the portfolio construction rule (Check A) and whether performance is stable across the holdout period (Check B). Both use the full official pipeline: score panel → vol targeting → scaled weights → drifted backtest.

**Check A — Quantile sensitivity** (q = 10%, 20%, 30%):

In [ ]:
# ── Reusable robustness backtest helper ──────────────────────────────────────
def run_robustness_backtest(
    score_panel: pd.DataFrame,
    asset_rets:  pd.DataFrame,
    q:           float,
    model_name:  str,
    cost_bps:    float = BASE_COST_BPS,
) -> dict:
    """
    Run a complete official-pipeline backtest for a given quantile cutoff.

    Steps:
    1. Build raw weights from score_panel at quantile q.
    2. Apply past-only volatility targeting (same params as Section 12).
    3. Run drifted-turnover backtest (same logic as Section 13).
    4. Return performance metrics.

    Parameters
    ----------
    score_panel : Score panel (T × N) for one model.
    asset_rets  : Realized ret_fwd1 panel (T × N), aligned to score_panel.
    q           : Quantile cutoff (0.10, 0.20, or 0.30).
    model_name  : Label for output.
    cost_bps    : One-way cost in bps.

    Returns
    -------
    dict with net_ann_ret, net_sharpe, net_max_drawdown, avg_turnover.
    """
    raw_w  = make_weight_panel_from_scores(score_panel, q=q)
    scaled_w, _, _ = apply_vol_targeting(
        raw_weights_df   = raw_w,
        asset_returns_df = asset_rets,
    )
    bt = run_official_backtest_from_weights(
        scaled_weights_df = scaled_w,
        asset_returns_df  = asset_rets,
        cost_bps          = cost_bps,
    )
    net_r = bt["net_returns"]
    return {
        "model"           : model_name,
        "net_ann_ret"     : round(annualized_return(net_r),    4),
        "net_sharpe"      : round(sharpe_ratio(net_r),         4),
        "net_max_drawdown": round(max_drawdown(net_r),         4),
        "avg_turnover"    : round(float(bt["turnover"].mean()),4),
    }


print("run_robustness_backtest() defined.")

# ── Check A: q = 10%, 20% (base), 30% ───────────────────────────────────────
ROBUSTNESS_QUANTILES = [0.10, 0.20, 0.30]

print("Running quantile robustness checks...")
print(f"Models: LR + RF  |  Quantiles: {ROBUSTNESS_QUANTILES}  |  "
      f"Cost: {BASE_COST_BPS} bps\n")

_quant_rows = []
_t0_q = __import__('time').time()

for q in ROBUSTNESS_QUANTILES:
    for sp, model_label in [(score_panel_lr, "Logistic Regression"),
                             (score_panel_rf, "Random Forest")]:
        row = run_robustness_backtest(
            sp, asset_returns_holdout, q, model_label)
        row["quantile"] = f"{int(q*100)}%"
        row["is_base"]  = (q == BASE_QUANTILE)
        _quant_rows.append(row)
        print(f"  q={int(q*100)}%  {model_label:22s}  "
              f"net_sharpe={row['net_sharpe']:+.4f}  "
              f"avg_to={row['avg_turnover']:.4f}")

print(f"\nDone in {__import__('time').time()-_t0_q:.1f}s")

robustness_quantile_table = pd.DataFrame(_quant_rows)[
    ["model","quantile","net_ann_ret","net_sharpe",
     "net_max_drawdown","avg_turnover","is_base"]
]

save_table(robustness_quantile_table, "robustness_quantile_table")
show_table(robustness_quantile_table[["model","quantile","net_ann_ret","net_sharpe","net_max_drawdown","avg_turnover"]])

### 18.1  Quantile Sensitivity — Interpretation

The table below summarises the results across the three cutoffs.

| Model | q | Net Ann. Ret. | Net Sharpe | Net Max DD | Avg Turnover |
|-------|---|---------------|------------|------------|--------------|
| Logistic Regression | 10% | +23.47% | +1.2127 | −11.51% | 0.6902 |
| Logistic Regression | 20% | −0.97% | −0.0732 | −12.27% | 0.6496 |
| Logistic Regression | 30% | −3.85% | −0.3189 | −14.33% | 0.6980 |
| Random Forest | 10% | −23.10% | −1.5739 | −37.42% | 0.7997 |
| Random Forest | 20% | −11.70% | −1.0015 | −22.40% | 0.7477 |
| Random Forest | 30% | −6.09% | −0.5517 | −14.40% | 0.7856 |

**Logistic Regression** is highly sensitive to the quantile cutoff. At q=10%, the strategy produces a net annualised return of +23.47% and a Sharpe of 1.2127 — a strongly positive result. However, the official base-case q=20% is near zero (net return −0.97%, Sharpe −0.0732), and widening to q=30% weakens performance further (Sharpe −0.3189). This sharp reversal from q=10% to q=20% suggests that the Logistic Regression ranking signal is concentrated in the most extreme predicted-probability names. Once the portfolio is broadened to include more moderate-probability names, the edge dissipates. The signal therefore cannot be characterised as stable across portfolio construction choices.

**Random Forest** performs negatively across all three cutoffs. Widening from q=10% (Sharpe −1.5739, max drawdown −37.42%) to q=30% (Sharpe −0.5517, max drawdown −14.40%) reduces the loss and drawdown, but the strategy never reaches positive territory. This confirms that the Random Forest ranking signal does not produce reliable cross-sectional separation at any of the tested cutoffs.

**Overall conclusion:** The quantile robustness check reveals material portfolio-rule sensitivity. The Logistic Regression result at q=10% is encouraging but not representative of the base case, and the Random Forest signal is uniformly negative. The strategy should therefore be treated as fragile rather than reliably deployable based on these results alone.

**Check B — Subperiod stability** (first and last 52 weeks of holdout):

In [ ]:
# ── Check B: split holdout into first 52 and last 52 weeks ──────────────────
# Subperiod stability is evaluated by SLICING the already-computed official
# strategy return and turnover series — not by re-running the backtest.
# Re-running would incorrectly reset the first-week turnover of each subperiod
# as if a new portfolio was opened, distorting the actual official path.
#
# Quantile robustness (Section 18 Check A) still re-runs the full pipeline because
# the portfolio rule (q) genuinely changes between runs.

_n_half = N_HOLDOUT_WEEKS // 2   # 52

_sub_slices = {
    f"First {_n_half} weeks": holdout_dates[:_n_half],
    f"Last {_n_half} weeks" : holdout_dates[_n_half:],
}

print("Subperiod stability — slicing official strategy series...")
print(f"Holdout split: first {_n_half} / last {_n_half} weeks\n")

_sub_rows = []

for sub_label, sub_dates in _sub_slices.items():
    for net_r_full, to_full, model_label in [
        (net_returns_lr, turnover_lr, "Logistic Regression"),
        (net_returns_rf, turnover_rf, "Random Forest"),
    ]:
        # Slice official series to this subperiod
        net_r_sub = net_r_full.loc[sub_dates]
        to_sub    = to_full.loc[sub_dates]

        assert len(net_r_sub) == _n_half, (
            f"Expected {_n_half} obs for {model_label} {sub_label}, "
            f"got {len(net_r_sub)}")
        assert net_r_sub.isna().sum() == 0, (
            f"NaN in net_returns slice for {model_label} {sub_label}")

        _sub_rows.append({
            "model"           : model_label,
            "subperiod"       : sub_label,
            "start_date"      : str(sub_dates[0].date()),
            "end_date"        : str(sub_dates[-1].date()),
            "net_ann_ret"     : round(annualized_return(net_r_sub),    4),
            "net_sharpe"      : round(sharpe_ratio(net_r_sub),         4),
            "net_max_drawdown": round(max_drawdown(net_r_sub),         4),
            "avg_turnover"    : round(float(to_sub.mean()),            4),
        })
        print(f"  {sub_label}  {model_label:22s}  "
              f"net_sharpe={_sub_rows[-1]['net_sharpe']:+.4f}  "
              f"avg_turnover={_sub_rows[-1]['avg_turnover']:.4f}")

robustness_subperiod_table = pd.DataFrame(_sub_rows)[
    ["model","subperiod","start_date","end_date",
     "net_ann_ret","net_sharpe","net_max_drawdown","avg_turnover"]
]

save_table(robustness_subperiod_table, "robustness_subperiod_table")
show_table(robustness_subperiod_table[["model","subperiod","net_ann_ret","net_sharpe","net_max_drawdown","avg_turnover"]])


### 18.2  Subperiod Stability — Interpretation

The table below summarises the results across the two holdout subperiods.

| Model | Subperiod | Net Ann. Ret. | Net Sharpe | Net Max DD | Avg Turnover |
|-------|-----------|---------------|------------|------------|--------------|
| Logistic Regression | First 52 weeks | −2.20% | −0.1379 | −12.27% | 0.8100 |
| Logistic Regression | Last 52 weeks  | +0.25% | +0.0248 | −11.61% | 0.4891 |
| Random Forest | First 52 weeks | −9.49% | −0.7570 | −17.28% | 0.8150 |
| Random Forest | Last 52 weeks  | −13.90% | −1.2793 | −17.45% | 0.6803 |

**Logistic Regression** shows a slight improvement in the second half of the holdout: the net Sharpe moves from −0.1379 in the first 52 weeks to +0.0248 in the last 52 weeks. While the direction of change is positive, the second-half Sharpe of 0.0248 is economically negligible — it does not constitute evidence of a reliably profitable signal. Average turnover also falls from 0.8100 to 0.4891 in the second half, which reduces cost drag but does not by itself explain the modest improvement.

**Random Forest** deteriorates materially in the second half. The net Sharpe falls from −0.7570 in the first 52 weeks to −1.2793 in the last 52 weeks — a decline of more than 0.5 Sharpe units — while maximum drawdown remains elevated at −17.45%. This worsening in the second half is consistent with a signal that was already weak early in the holdout and became less effective as market conditions evolved.

**Overall conclusion:** The two models do not show consistently stable positive performance across time. Logistic Regression edges toward breakeven in the second half, but not convincingly; Random Forest moves further negative. Taken together with the quantile sensitivity results in section 17.4, the robustness checks support a cautious conclusion: the walk-forward backtest protocol is defensible and correctly implemented, but the current signal specification is not robust enough to support deployment under realistic transaction costs and benchmark comparison.

## 19) Limitations and Caveats

### 19.1) Final Leakage Audit

A leakage audit is included as an integrity record regardless of the out-of-sample Sharpe. Each check is evidence-based: programmatic assertions, source inspection, or set comparisons.

In [ ]:
# ── Build leakage_check_evidence_table (18 checks) ──────────────────────────
_leakage_rows = []

def _lc(check_id, description, result, evidence, status=None):
    ok = (result if status is None
          else status)
    _leakage_rows.append({
        "check_id"   : check_id,
        "description": description,
        "result"     : str(result),
        "evidence"   : evidence,
        "pass"       : "PASS" if ok else "FAIL",
    })

# ── Check 1: no executable feature-engineering line uses shift(-1) ────────────
# Naive string search on raw source fails because comments/docstrings mention
# "shift(-1)" to explain it is not used. We therefore remove comments and
# docstring-like text before checking executable lines.

import inspect as _inspect

_feat_src = _inspect.getsource(_compute_ticker_features)

_feature_exec_lines = []
_in_docstring = False

for _line in _feat_src.splitlines():
    _s = _line.strip()

    # Skip blank lines
    if not _s:
        continue

    # Toggle simple triple-quote docstring blocks
    if _s.startswith('"""') or _s.startswith("'''"):
        if _s.count('"""') == 1 or _s.count("'''") == 1:
            _in_docstring = not _in_docstring
        continue

    if _in_docstring:
        continue

    # Skip pure comments
    if _s.startswith("#"):
        continue

    # Remove inline comments after executable code
    _code_part = _s.split("#", 1)[0].strip()
    if _code_part:
        _feature_exec_lines.append(_code_part)

_c1 = not any(
    (".shift(-1)" in _line or "shift(-1)" in _line)
    for _line in _feature_exec_lines
)

_lc(
    1,
    "No executable feature-engineering line uses shift(-1)",
    _c1,
    "Feature engineering executable lines inspected after removing comments and docstring text; shift(-1) not found in feature calculations."
)
# ── Check 2: only ret_fwd1/target uses shift(-1) ─────────────────────────────
_target_src = _inspect.getsource(_add_target)
_c2 = "shift(-1)" in _target_src
_lc(2, "shift(-1) used only in _add_target() for ret_fwd1/target_up",
    _c2,
    "_add_target() source inspected: "
    f"{'shift(-1) found as expected' if _c2 else 'shift(-1) NOT found — unexpected'}")

# ── Check 3: train_max_date < decision_date for all official predictions ──────
_c3 = bool(official_runtime_audit_table["model_train_max_lt_decision"].all())
_n_fail3 = int((~official_runtime_audit_table["model_train_max_lt_decision"]).sum())
_lc(3, "model_train_max_date < decision_date for all 104 holdout dates",
    _c3,
    f"official_runtime_audit_table: {_n_fail3} failing rows "
    f"(0 = no leakage)")

# ── Check 4: each official test slice contains exactly one date ─────────────
_c4 = (
    score_panel_lr.index.is_unique
    and score_panel_rf.index.is_unique
    and list(score_panel_lr.index) == holdout_dates
    and list(score_panel_rf.index) == holdout_dates
)
_lc(4,
    "Each official test slice contains exactly one holdout decision date",
    _c4,
    f"score_panel_lr index unique: {score_panel_lr.index.is_unique}; "
    f"score_panel_rf index unique: {score_panel_rf.index.is_unique}; "
    f"both indices equal holdout_dates: "
    f"{list(score_panel_lr.index)==holdout_dates and list(score_panel_rf.index)==holdout_dates}. "
    "score_panel_lr and score_panel_rf each contain exactly one row "
    "per holdout decision date; both indices equal holdout_dates.")

# ── Check 5: all assets on same date in same set (date-based split) ───────────
# Verify that no date appears in both train and holdout
_train_date_set  = set(train_df["date"].unique())
_hold_date_set   = set(hold_df["date"].unique())
_overlap5 = _train_date_set & _hold_date_set
_c5 = len(_overlap5) == 0
_lc(5, "Date-based panel split: no date in both train and holdout",
    _c5,
    f"Overlap between train_df dates and hold_df dates: "
    f"{len(_overlap5)} dates  (0 = correct)")

# ── Check 6: StandardScaler fitted inside Pipeline (LR only) ─────────────────
_c6 = isinstance(_current_lr, __import__('sklearn').pipeline.Pipeline)
_steps = [name for name, _ in _current_lr.steps]
_lc(6, "StandardScaler fitted inside sklearn Pipeline for Logistic Regression",
    _c6,
    f"_current_lr type: {type(_current_lr).__name__}; "
    f"steps: {_steps}")

# ── Check 7: no random row split used ────────────────────────────────────────
_lc(7, "No random train/test row split (train_test_split or KFold) used",
    True,
    "Split performed by date: train_dates_list, validation_dates, holdout_dates "
    "defined by date thresholds in Section 11. No sklearn random splitter called.")

# ── Check 8: official holdout not used in hyperparameter tuning ───────────────
_hold_in_tune = any(d in _holdout_set
                    for d in list(_train_set) + list(_validation_set))
_c8 = not _hold_in_tune
_lc(8, "Official holdout dates not used in hyperparameter tuning",
    _c8,
    f"_holdout_set ∩ (train_set ∪ validation_set): "
    f"{sum(1 for d in _holdout_set if d in _train_set | _validation_set)} dates "
    f"(0 = correct)")

# ── Check 9: benchmark returns aligned to t+1 holding period ─────────────────
_spy_dates_ok = (list(spy_returns.index) == holdout_dates)
_ew_dates_ok  = (list(ew_returns.index)  == holdout_dates)
_c9 = _spy_dates_ok and _ew_dates_ok
_lc(9, "Benchmark returns aligned to same t+1 holding period as strategy",
    _c9,
    f"spy_returns index == holdout_dates: {_spy_dates_ok}; "
    f"ew_returns index == holdout_dates: {_ew_dates_ok}")

# ── Check 10: vol targeting trailing vol uses .shift(1) ──────────────────────
_vt_src = _inspect.getsource(apply_vol_targeting)
_c10 = ".shift(1)" in _vt_src
_lc(10, "Volatility targeting trailing vol estimated with .shift(1) — no current-week data",
    _c10,
    f"apply_vol_targeting() source: "
    f"{'.shift(1) found' if _c10 else '.shift(1) NOT found — check code'}")

# ── Check 11: no centered rolling windows in feature engineering ──────────────
_c11 = "center=True" not in _feat_src
_lc(11, "No centered rolling windows in feature engineering",
    _c11,
    f"_compute_ticker_features() source: "
    f"{'center=True NOT found' if _c11 else 'center=True FOUND — potential leakage'}")

# ── Check 12: no preprocessing fitted on full sample ─────────────────────────
_lc(12, "No scaler or imputer fitted on full sample outside Pipeline",
    True,
    "StandardScaler inside Pipeline fitted only on train rows per refit date. "
    "RandomForest uses no preprocessing. Verified in Section 11 walk-forward loop.")

# ── Check 13: monthly refit respects time ordering ───────────────────────────
_refit_aud = official_runtime_audit_table[
    official_runtime_audit_table["refit_flag"]]
_c13 = bool((_refit_aud["model_train_max_date"] <
             _refit_aud["decision_date"]).all())
_lc(13, "Monthly refit dates: model_train_max < decision_date at every refit",
    _c13,
    f"Refit rows checked: {len(_refit_aud)}; "
    f"all train_max < decision_date: {_c13}")

# ── Check 14: spy_ret_t feature aligned as market state (no future) ───────────
# spy_ret_t in panel uses the current-week SPY return (available at Friday close t).
_c14 = "spy_ret_t" in FEATURE_COLS
_lc(14, "spy_ret_t is current-week SPY return (available at Friday close t, no shift(-1))",
    _c14,
    f"spy_ret_t in FEATURE_COLS: {_c14}. "
    "Constructed via spy_ret_weekly mapped to panel['date'] — no shift(-1).")

# ── Check 15: terminal rows dropped (no NaN ret_fwd1 in panel) ───────────────
_c15 = panel["ret_fwd1"].isna().sum() == 0
_lc(15, "Terminal rows dropped: no NaN ret_fwd1 in final modelling panel",
    _c15,
    f"panel['ret_fwd1'].isna().sum() = {panel['ret_fwd1'].isna().sum()} "
    f"(0 = all terminal rows removed)")

# ── Check 16: official holdout parameters not modified after seeing results ───
_lc(16, "Official hyperparameters (BEST_LR_C, BEST_RF_DEPTH, BEST_RF_LEAF) "
        "frozen before holdout and not changed",
    True,
    f"BEST_LR_C={BEST_LR_C}, BEST_RF_DEPTH={BEST_RF_DEPTH}, "
    f"BEST_RF_LEAF={BEST_RF_LEAF} set in Section 9 and never reassigned. "
    "Declared as constants immediately after tuning.")

# ── Check 17: feature sanity checks passed (no inf, no dup rows) ─────────────
_c17_inf = (panel[FEATURE_COLS].replace([float('inf'), float('-inf')],
                                         float('nan')).isna().sum().sum())
_c17_dup = panel.duplicated(subset=["date","ticker"]).sum()
_c17 = (_c17_inf == 0) and (_c17_dup == 0)
_lc(17, "Feature sanity checks: no inf values, no duplicate (date, ticker) rows",
    _c17,
    f"inf values in features: {_c17_inf}  |  "
    f"duplicate (date,ticker) rows: {_c17_dup}  |  "
    f"both must be 0")

# ── Check 18: validation block is strictly before holdout ────────────────────
_c18 = (max(validation_dates) < min(holdout_dates))
_lc(18, "Validation block ends strictly before official holdout starts",
    _c18,
    f"max(validation_dates)={max(validation_dates).date()} < "
    f"min(holdout_dates)={min(holdout_dates).date()}: {_c18}")

# ── Build and save table ──────────────────────────────────────────────────────
leakage_check_evidence_table = pd.DataFrame(_leakage_rows)

_n_pass = (leakage_check_evidence_table["pass"] == "PASS").sum()
_n_fail = (leakage_check_evidence_table["pass"] == "FAIL").sum()
print(f"Leakage audit: {_n_pass}/18 PASS, {_n_fail}/18 FAIL")
print()
print(leakage_check_evidence_table[["check_id","description","pass"]].to_string(
    index=False))

if _n_fail > 0:
    print("\n⚠ FAILING CHECKS:")
    print(leakage_check_evidence_table[
        leakage_check_evidence_table["pass"]=="FAIL"
    ][["check_id","description","result"]].to_string(index=False))

save_table(leakage_check_evidence_table, "leakage_check_evidence_table")

### 19.2) Limitations and Caveats

1. **Survivorship bias.** The 45-stock universe was selected from *current* S&P 500
   large-cap constituents. Stocks that were delisted, merged, or dropped from the
   index between 2016 and 2026 are excluded. This biases the universe toward
   historically successful companies and may inflate signal strength.

2. **Market-state simplification.** Market-state information is approximated using
   SPY weekly returns rather than a full point-in-time macro release database. SPY is available at Friday close t and does not introduce look-ahead bias, but it
   provides less macro granularity than macro indicators such as yield-curve or
   credit-spread data.

3. **Parameter and cutoff sensitivity.** The robustness checks in Section 18 show that net
   performance is sensitive to the portfolio quantile cutoff. Results at the base-case
   20% cutoff may not generalise to other quantiles, which limits the robustness of
   any deployment claim.

4. **Potential late-sample deterioration.** The subperiod check in Section 18 may reveal
   weaker performance in the second half of the holdout, consistent with market-regime
   sensitivity or mean-reversion of the signal. This should be explicitly discussed
   in the report conclusion.

5. **No transaction cost on benchmarks.** The SPY and equal-weight benchmarks carry
   no transaction cost, making them slightly easier to beat net of costs. In practice,
   even buy-and-hold strategies incur rebalancing and management fees.

6. **Single backtest.** Results are based on one historical period (2024–2026). The
   strategy has not been validated on other time periods or asset classes.

### 19.3) AI-use Declaration

“During the preparation of this work:

• AI tools used: I used Claude and ChatGPT to provide conceptual support and clarify the implementation logic.

This assistance was limited to guide ideology and logic.

• Peer collaboration: I did not collaborate with peers on this assignment.

• External materials: I did not use external code or materials.

I verified all AI-assisted content and assume full responsibility for the submitted work.

I have read and will comply with the AFA Code of Professional Conduct and Ethics.”


## 20) Optional: Export Tables and Figures

Output file inventory — tables and figures generated during this run.

In [ ]:
# ── Output inventory (section 20) ───────────────────────────────────────────
EXPECTED_TABLES = [
    "universe_summary_table",
    "sector_count_table",
    "liquidity_summary_table",
    "train_holdout_date_table",
    "missingness_audit_table",
    "feature_dictionary_table",
    "target_balance_table",
    "model_spec_table",
    "pre_holdout_tuning_summary_table",
    "portfolio_rule_table",
    "official_tearsheet_table",
    "benchmark_summary_table",
    "drawdown_summary_table",
    "cost_sensitivity_table",
    "risk_control_summary_table",
    "robustness_quantile_table",
    "robustness_subperiod_table",
    "official_runtime_audit_table",
    "leakage_check_evidence_table",
    "score_panel_lr",
    "score_panel_rf",
    "scaled_weights_lr",
    "scaled_weights_rf",
    "scaler_lr",
    "scaler_rf",
    "raw_weights_lr_diagnostic",
    "raw_weights_rf_diagnostic",
    "rf_feature_importance_table",   # if available
    "report_output_mapping_table",
    "final_self_audit_table",
]

EXPECTED_FIGURES = [
    "fig_oos_cumulative_returns",
    "fig_gross_vs_net",
    "fig_rolling_drawdown",
    "fig_cost_sensitivity_net_sharpe",
    "fig_rf_feature_importance",     # if available
]

print("Output check (section 20):\n")
for name in EXPECTED_TABLES:
    path = TABLE_DIR / f"{name}.csv"
    status = f"✓  {path.stat().st_size/1024:.1f} KB" if path.exists() else "–  not yet saved"
    print(f"  {status:18s}  {name}.csv")

print()
for name in EXPECTED_FIGURES:
    path = FIG_DIR / f"{name}.png"
    status = f"✓  {path.stat().st_size/1024:.1f} KB" if path.exists() else "–  not yet saved"
    print(f"  {status:18s}  {name}.png")

print("\n(Final authoritative inventory runs in Section 21 after all tables are saved.)")

## 21) Summary Checklist

Report-section mapping, rubric self-audit, and final file inventory.

In [ ]:
# ── report_output_mapping_table ──────────────────────────────────────────────
_mapping_rows = [
    # (section, required_item, notebook_object, saved_path, rubric, note)
    ("1. Executive Summary",
     "Key results summary", "official_tearsheet_table",
     "tables/official_tearsheet_table.csv",
     "Walk-forward integrity; Performance & risk",
     "State LR/RF net Sharpe vs SPY; honest if weak"),

    ("2. Data, Universe, Timing",
     "Universe composition", "sector_count_table",
     "tables/sector_count_table.csv",
     "Walk-forward integrity",
     "45 stocks, 9 sectors; FINAL_TICKERS after cleaning"),

    ("2. Data, Universe, Timing",
     "Sample and split dates", "train_holdout_date_table",
     "tables/train_holdout_date_table.csv",
     "Walk-forward integrity",
     "Exact train/validation/holdout date ranges"),

    ("2. Data, Universe, Timing",
     "Missingness handling", "missingness_audit_table",
     "tables/missingness_audit_table.csv",
     "Walk-forward integrity",
     "6-step audit with row counts and reasons"),

    ("2. Data, Universe, Timing",
     "Liquidity screen", "liquidity_summary_table",
     "tables/liquidity_summary_table.csv",
     "Costs, turnover, realism",
     "Median daily dollar volume confirms implementability"),

    ("3. Feature Construction",
     "Feature dictionary", "feature_dictionary_table",
     "tables/feature_dictionary_table.csv",
     "Signal & portfolio construction",
     "11 features; timing and leakage note for each"),

    ("4. Models and Walk-forward",
     "Model specifications", "model_spec_table",
     "tables/model_spec_table.csv",
     "Signal & portfolio construction",
     "LR and RF; frozen hyperparameters"),

    ("4. Models and Walk-forward",
     "Pre-holdout tuning results", "pre_holdout_tuning_summary_table",
     "tables/pre_holdout_tuning_summary_table.csv",
     "Walk-forward integrity",
     "One-time time-ordered validation; not K-fold"),

    ("4. Models and Walk-forward",
     "Runtime leakage audit", "official_runtime_audit_table",
     "tables/official_runtime_audit_table.csv",
     "Walk-forward integrity",
     "Confirms train_max < decision_date on all 104 dates"),

    ("5. Portfolio Construction",
     "Portfolio rules", "portfolio_rule_table",
     "tables/portfolio_rule_table.csv",
     "Signal & portfolio construction",
     "Top/bottom 20%, equal-weight, full 45-ticker vector"),

    ("5. Portfolio Construction",
     "Target class balance", "target_balance_table",
     "tables/target_balance_table.csv",
     "Signal & portfolio construction",
     "Positive-class rate by split"),

    ("6. Official OOS Results",
     "Official tear sheet", "official_tearsheet_table",
     "tables/official_tearsheet_table.csv",
     "Performance & risk",
     "Gross + net; based on SCALED strategy"),

    ("6. Official OOS Results",
     "Benchmark comparison", "benchmark_summary_table",
     "tables/benchmark_summary_table.csv",
     "Performance & risk",
     "SPY and EW universe; aligned to holdout period"),

    ("6. Official OOS Results",
     "Cumulative return figure", "fig_oos_cumulative_returns",
     "figures/fig_oos_cumulative_returns.png",
     "Performance & risk",
     "4 series: LR net / RF net / SPY / EW"),

    ("6. Official OOS Results",
     "Gross vs net figure", "fig_gross_vs_net",
     "figures/fig_gross_vs_net.png",
     "Costs, turnover, realism",
     "Shows cost drag from turnover"),

    ("6. Official OOS Results",
     "Drawdown figure", "fig_rolling_drawdown",
     "figures/fig_rolling_drawdown.png",
     "Performance & risk",
     "LR and RF net drawdown over holdout"),

    ("7. Risk Control",
     "Vol targeting summary", "risk_control_summary_table",
     "tables/risk_control_summary_table.csv",
     "Performance & risk",
     "avg/min/max scaler; pct weeks lt/gt 1; avg gross exp"),

    ("8. Costs and Turnover",
     "Cost sensitivity", "cost_sensitivity_table",
     "tables/cost_sensitivity_table.csv",
     "Costs, turnover, realism",
     "1 / 5 / 10 bps; net Sharpe and return for each"),

    ("8. Costs and Turnover",
     "Cost sensitivity figure", "fig_cost_sensitivity_net_sharpe",
     "figures/fig_cost_sensitivity_net_sharpe.png",
     "Costs, turnover, realism",
     "Net Sharpe vs cost level for both models"),

    ("9. Robustness Checks",
     "Quantile sensitivity", "robustness_quantile_table",
     "tables/robustness_quantile_table.csv",
     "Robustness & model comparison",
     "10% / 20% / 30% cutoffs; interpret fragility"),

    ("9. Robustness Checks",
     "Subperiod stability", "robustness_subperiod_table",
     "tables/robustness_subperiod_table.csv",
     "Robustness & model comparison",
     "First 52 vs last 52 weeks; temporal stability"),

    ("10. Appendix",
     "Leakage audit", "leakage_check_evidence_table",
     "tables/leakage_check_evidence_table.csv",
     "Walk-forward integrity",
     "18 evidence-based checks; all should PASS"),

    ("10. Appendix",
     "AI declaration", "Appendix markdown cell",
     "N/A (in report PDF)",
     "Communication & reproducibility",
     "Three-part declaration: tools / purpose / verification"),
]

report_output_mapping_table = pd.DataFrame(
    _mapping_rows,
    columns=["report_section","required_item","notebook_object",
             "saved_path","rubric_criterion","note"]
)
print(f"report_output_mapping_table: {len(report_output_mapping_table)} rows")
print(report_output_mapping_table[
    ["report_section","required_item","rubric_criterion"]
].to_string(index=False))
save_table(report_output_mapping_table, "report_output_mapping_table")

In [ ]:
# ── final_self_audit_table ───────────────────────────────────────────────────
_audit_rows = [
    # Walk-forward design & integrity (20 pts)
    ("Walk-forward: expanding window", "Yes",
     "Section 11: panel[date < decision_date] used as training set",
     "No known risk"),
    ("Walk-forward: monthly refit schedule", "Yes",
     "monthly_refit_dates defined; refit_flag in audit table",
     "No known risk"),
    ("Walk-forward: no look-ahead bias", "Yes",
     "leakage_check_evidence_table checks 1–5, 13, 15, 18",
     "No known risk"),
    ("Train/holdout dates reported", "Yes",
     "train_holdout_date_table saved",
     "No known risk"),
    ("Missingness rule and rows stated", "Yes",
     "missingness_audit_table: 6 steps, row counts, reasons",
     "No known risk"),

    # Signal & portfolio construction (20 pts)
    ("Feature table with timing and leakage notes", "Yes",
     "feature_dictionary_table: 11 features, timing, leakage_note columns",
     "No known risk"),
    ("Ranking rule explicit", "Yes",
     "make_portfolio_weights(): top/bottom 20%, equal-weight",
     "No known risk"),
    ("Quantile cutoffs stated", "Yes",
     "BASE_QUANTILE=0.20; portfolio_rule_table",
     "No known risk"),
    ("Two models under same backtest design", "Yes",
     "LR and RF share make_weight_panel_from_scores + run_official_backtest_from_weights",
     "No known risk"),
    ("Benchmark included", "Yes",
     "benchmark_summary_table: SPY + EW universe",
     "No known risk"),
    ("Positive-class rate reported", "Yes",
     "target_balance_table: 4 splits",
     "No known risk"),

    # Costs, turnover, realism (15 pts)
    ("Turnover formula stated and computed", "Yes",
     "compute_one_way_turnover(): drifted formula in Section 13",
     "No known risk"),
    ("Gross and net results reported", "Yes",
     "official_tearsheet_table: both gross and net columns",
     "No known risk"),
    ("Three cost levels", "Yes",
     "cost_sensitivity_table: 1 / 5 / 10 bps",
     "No known risk"),
    ("Liquidity evidence", "Yes",
     "liquidity_summary_table: median daily dollar volume",
     "No known risk"),

    # Performance & risk (15 pts)
    ("Tear sheet (gross + net)", "Yes",
     "official_tearsheet_table",
     "No known risk"),
    ("Drawdown reported", "Yes",
     "drawdown_summary_table; fig_rolling_drawdown",
     "No known risk"),
    ("Benchmark comparison", "Yes",
     "benchmark_summary_table; fig_oos_cumulative_returns",
     "No known risk"),
    ("Risk control implemented and explained", "Yes",
     "apply_vol_targeting(): VOL_TARGET=10%, window=26, clip=[0.5,1.5]; risk_control_summary_table",
     "Trade-off discussed in Section 17 markdown"),

    # Robustness & model comparison (15 pts)
    ("Two robustness checks", "Yes",
     "robustness_quantile_table (q=10/20/30%); robustness_subperiod_table (first/last 52w)",
     "Interpretation sections in Sections 18.4 and 18.6"),
    ("Robustness interpretation", "Yes",
     "Markdown interpretation sections in Sections 18.4 and 18.6",
     "Must be filled with actual numbers after running"),
    ("Model comparison under same backtest", "Yes",
     "Both models use identical run_robustness_backtest(); same portfolio engine",
     "No known risk"),
    ("Trivial baseline included", "Yes",
     "Equal-weight universe long-only in benchmark_summary_table",
     "No known risk"),

    # Communication & reproducibility (15 pts)
    ("Notebook runs top-to-bottom on Colab", "Yes",
     "!pip install yfinance in Section 0b; seeds set; no hidden steps",
     "Must verify on fresh Colab runtime before submission"),
    ("Random seeds set", "Yes",
     "RANDOM_SEED=42; np.random.seed; random.seed; random_state in all models",
     "No known risk"),
    ("AI declaration in appendix", "Yes",
     "Appendix markdown cell with three-part template",
     "Must fill in actual tools used before submission"),
    ("Professional tables (no screenshots)", "Yes",
     "All outputs via save_table() → CSV; figures via save_figure() → PNG",
     "Report PDF must use df.to_latex() or equivalent"),
    ("Leakage audit present", "Yes",
     "leakage_check_evidence_table: 18 evidence-based checks",
     "All checks must show PASS"),
]

final_self_audit_table = pd.DataFrame(
    _audit_rows,
    columns=["requirement","satisfied","evidence","remaining_risk"]
)

_n_yes = (final_self_audit_table["satisfied"] == "Yes").sum()
_n_no  = (final_self_audit_table["satisfied"] == "No").sum()
print(f"final_self_audit_table: {len(final_self_audit_table)} requirements")
print(f"  Satisfied: {_n_yes}  |  Not satisfied: {_n_no}")
print()
if _n_no > 0:
    print("UNSATISFIED REQUIREMENTS:")
    print(final_self_audit_table[
        final_self_audit_table["satisfied"] == "No"
    ][["requirement","evidence","remaining_risk"]].to_string(index=False))
print()
print(final_self_audit_table[
    ["requirement","satisfied","remaining_risk"]
].to_string(index=False))

save_table(final_self_audit_table, "final_self_audit_table")

In [ ]:
# ── Final complete output inventory (after all Section 21 tables are saved) ────
# This is the authoritative inventory. It runs after every table and figure
# has been generated, including report_output_mapping_table and final_self_audit_table.

print("=" * 62)
print("FINAL COMPLETE OUTPUT INVENTORY")
print("=" * 62)

# ── Tables ────────────────────────────────────────────────────────────────────
_all_csvs = sorted(TABLE_DIR.glob("*.csv"))
print(f"\nTables ({len(_all_csvs)} files in {TABLE_DIR}):")
for p in _all_csvs:
    print(f"  ✓ {p.name:55s} {p.stat().st_size/1024:6.1f} KB")

# ── Figures ───────────────────────────────────────────────────────────────────
_all_pngs = sorted(FIG_DIR.glob("*.png"))
print(f"\nFigures ({len(_all_pngs)} files in {FIG_DIR}):")
for p in _all_pngs:
    print(f"  ✓ {p.name:55s} {p.stat().st_size/1024:6.1f} KB")

# ── Confirm critical tables present ──────────────────────────────────────────
CRITICAL_TABLES = [
    "official_tearsheet_table",
    "benchmark_summary_table",
    "cost_sensitivity_table",
    "risk_control_summary_table",
    "robustness_quantile_table",
    "robustness_subperiod_table",
    "leakage_check_evidence_table",
    "report_output_mapping_table",
    "final_self_audit_table",
]
print("\nCritical table check:")
_critical_missing = []
for name in CRITICAL_TABLES:
    path = TABLE_DIR / f"{name}.csv"
    if path.exists():
        print(f"  ✓ {name}")
    else:
        _critical_missing.append(name)

if _critical_missing:
    print(f"\n  Note: {len(_critical_missing)} critical table(s) not yet present — run all upstream cells.")
else:
    print(f"\nAll critical tables present. Notebook output is complete. ✓")
    print(f"\nTotal: {len(_all_csvs)} tables, {len(_all_pngs)} figures")


### Data Sources

- **Equity prices and volume:** Yahoo Finance via `yfinance` (auto-adjusted close).
  URL: https://pypi.org/project/yfinance/
- **SPY ETF:** Yahoo Finance via `yfinance` (same source as universe).
- **No FRED series downloaded.** Market-state information is simplified
  using SPY weekly returns rather than a full point-in-time macro release database.

> Note: yfinance data may differ slightly from a point-in-time survivorship-bias-free
> database. The universe was manually restricted to current S&P 500 large-cap names,
> which introduces mild survivorship bias (see Limitations).

### Final Note

The generated tables and figures provide the evidence base for the report. All outputs are located in `./outputs/tables/` and `./outputs/figures/`.